# 섹터 1 — 공 굴리기 환경: DIDEN_World 를 축소판으로 처음부터

공에 토크를 걸어 **목적지까지 굴려 가서 멈추는** 정책을 학습시킵니다. 과제는 단순하지만, 만드는 구조는 DIDEN_World 가 Spider 를 걷게 할 때 쓰는 구조와 같습니다. 이름도 일부러 같게 짓습니다.

```
            ┌────────────────────────── ManagerBasedRlEnv.step(action) ──────────────────────────┐
 action ──▶ │ ActionManager ─▶ [ 물리 × decimation ] ─▶ Termination ─▶ Reward ─▶ (리셋: Event) ─▶ Command ─▶ Observation │ ──▶ obs, reward, terminated, time_outs, extras
            └───────────────────────────────────────▲─────────────────────────────────────────────┘
                                                    │ SimAdapter (엔진과 닿는 유일한 창구)
                                              BatchedSim (mujoco_warp, GPU, 환경 N 개)
 PPO:  OnPolicyRunner ─ collect(RolloutStorage) ─ compute_returns(GAE) ─ update(ActorCritic)
```

| Part | 만드는 것 | DIDEN_World 에서 대응하는 파일 |
|---|---|---|
| 1 | `BatchedSim` — GPU 병렬 시뮬레이션 | `mjlab.sim.Simulation` (외부 라이브러리), `diden_mjlab/sim/sim.py` |
| 2 | 쿼터니언 연산 (torch) | `diden_common/math.py` |
| 3 | `SimAdapter` — 상태 읽기/쓰기 | `diden_common/sim_adapter.py`, `sim_adapter_ports.py`, `diden_mjlab/sim_adapter.py` |
| 4 | term 설정과 `ManagerBase` | `diden_common/manager_term_cfg.py`, `diden_env/managers/manager_base.py` |
| 5 | `CommandManager` — 목적지 | `diden_env/managers/command_manager.py`, `diden_env/mdp/commands/` |
| 6 | `ActionManager` — 토크 | `diden_env/managers/action_manager.py`, `diden_env/mdp/actions/joint_actions.py` |
| 7 | `ObservationManager` | `diden_env/managers/observation_manager.py` |
| 8 | `RewardManager` | `diden_env/managers/reward_manager.py` |
| 9 | `TerminationManager` | `diden_env/managers/termination_manager.py` |
| 10 | `EventManager` — 리셋과 도메인 랜덤화 | `diden_env/managers/event_manager.py`, `diden_env/mdp/events/` |
| 11 | `ManagerBasedRlEnv` — 전부 묶기 | `diden_env/env/manager_based_env.py`, `manager_based_rl_env.py`, `diden_mjlab/env/mjwarp_env.py` |
| 12 | `RunningNormalization`, `ActorCritic` | `diden_rl/modules/utils.py`, `actor_critic.py` |
| 13 | `RolloutStorage`, GAE | `diden_rl/storage/rollout_storage.py` |
| 14 | `PPO` | `diden_rl/algorithms/ppo.py` |
| 15 | `OnPolicyRunner`, 학습 | `diden_rl/runners/on_policy_runner.py`, `tools/engines/mjwarp/train.py` |
| 16 | 평가와 영상 | `tools/engines/mjwarp/play.py` |
| 17 | ONNX 내보내기와 "Core 처럼" 실행하기 | `tools/deploy/export_merged_onnx.py`, `diden_env/deploy/`; Core 의 `rl_controller/` |

**실제 코드와 다른 점** 을 미리 적어 둡니다. 이 노트북은 구조를 손에 익히는 것이 목적이라 다음을 뺐습니다. 각 Part 끝에서 "실제로는 여기에 무엇이 더 있는가" 를 다시 짚습니다.

- 로봇이 없습니다. 관절, PD 액추에이터, 접촉 센서, 자석, 보행 위상(gait)은 섹터 2 에서 다룹니다.
- 상태 추정기 네트워크(`PPOEstimator`), 대칭 증강, 커리큘럼, 관측 이력·지연, 액션 지연이 없습니다.
- World 는 엔진 세 개(MuJoCo-Warp, IsaacLab, RaiSim)를 `SimAdapter` 뒤에 숨깁니다. 여기서는 MuJoCo-Warp 하나만 붙입니다.

**준비물**: 섹터 0 을 끝냈다고 가정합니다. NVIDIA GPU 가 필요합니다. 첫 실행 때 Warp 가 물리 커널을 컴파일하느라 1~2분 걸리고, 그 뒤로는 캐시를 씁니다.


In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")

import math
import time
from dataclasses import dataclass, field, fields
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
import torch.nn as nn
import mujoco
import warp as wp
import mujoco_warp as mjw
import mediapy as media
import matplotlib.pyplot as plt

wp.config.quiet = True
np.set_printoptions(precision=4, suppress=True, linewidth=140)
torch.set_printoptions(precision=4, sci_mode=False, linewidth=140)

assert torch.cuda.is_available(), "이 노트북은 NVIDIA GPU 가 필요합니다."
DEVICE = "cuda:0"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "DIDEN_Asset" / "object").is_dir())
ASSET = ROOT / "DIDEN_Asset"
print("mujoco", mujoco.__version__, "| torch", torch.__version__, "| warp", wp.__version__, "|", torch.cuda.get_device_name(0))


In [ ]:
# 채점·렌더링 도우미 (주어진 코드)

def _np(x):
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)


def check(name, cond):
    cond = bool(cond)
    print(("✅ " if cond else "❌ ") + name)
    assert cond, name


def check_close(name, got, want, atol=1e-5, rtol=0.0):
    got, want = _np(got).astype(float), _np(want).astype(float)
    ok = got.shape == want.shape and np.allclose(got, want, atol=atol, rtol=rtol)
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"\n   got  = {got}\n   want = {want}"))
    assert ok, name


def make_camera(lookat=(0, 0, 0.2), distance=9.0, azimuth=90, elevation=-55):
    cam = mujoco.MjvCamera()
    cam.lookat[:] = lookat
    cam.distance, cam.azimuth, cam.elevation = distance, azimuth, elevation
    return cam


### 씬 (섹터 0 의 Part 8 에서 직접 만든 것)

섹터 0 에서 만든 `build_ball_scene` 을 그대로 가져옵니다. 달라진 점은 세 가지입니다.

- 바닥에 격자 무늬를 입혔습니다.
- **바닥 마찰을 낮게(0.05)** 둡니다. 두 geom 의 유효 마찰은 큰 쪽이므로, 이렇게 해 두면 공의 마찰계수가 곧 유효 마찰이 됩니다. Part 10 에서 공의 마찰을 환경마다 다르게 랜덤화합니다.
- 접촉 풀이 설정을 World 의 Spider 설정에 가깝게 맞췄습니다: 타원 마찰 원뿔(`cone="elliptic"`), 솔버 반복 10회. World 는 선탐색(line search) 반복도 10회로 줄여 쓰는데(`diden_mjlab/sim/sim.py` 의 `get_default_sim_cfg`), 이 씬에서 그렇게 줄이면 "linesearch iterations limit reached" 경고가 나와서 MuJoCo 기본값(50회)을 그대로 둡니다. 반복 횟수는 줄일수록 빠르지만 접촉 풀이가 덜 수렴합니다.


In [ ]:
BALL_RADIUS = 0.2
BALL_MASS = 1.0
GRAVITY = 9.81


def build_ball_scene(ball_mass: float = BALL_MASS, ball_friction: float = 1.0, floor_friction: float = 0.05,
                     timestep: float = 0.005) -> mujoco.MjSpec:
    G = mujoco.mjtGeom
    spec = mujoco.MjSpec()
    spec.modelname = "ball_scene"
    spec.option.timestep = timestep
    spec.option.cone = mujoco.mjtCone.mjCONE_ELLIPTIC
    spec.option.iterations = 10
    wb = spec.worldbody
    wb.add_light(name="sun", pos=[0, 0, 8], dir=[0, 0, -1], type=mujoco.mjtLightType.mjLIGHT_DIRECTIONAL)
    spec.add_texture(name="grid", type=mujoco.mjtTexture.mjTEXTURE_2D, builtin=mujoco.mjtBuiltin.mjBUILTIN_CHECKER,
                     width=256, height=256, rgb1=[0.22, 0.24, 0.28], rgb2=[0.30, 0.32, 0.37])
    mat = spec.add_material(name="grid", texrepeat=[1, 1])
    mat.textures[mujoco.mjtTextureRole.mjTEXROLE_RGB] = "grid"
    wb.add_geom(name="floor", type=G.mjGEOM_PLANE, size=[0, 0, 0.05], friction=[floor_friction, 0.005, 0.0001], material="grid")

    child = mujoco.MjSpec.from_file(str(ASSET / "object" / "sphere.xml"))
    spec.attach(child, prefix="ball/", frame=wb.add_frame(pos=[0, 0, BALL_RADIUS]))
    ball = spec.body("ball/sphere")
    ball.add_freejoint(name="ball/root")
    for g in ball.geoms:
        if g.contype:
            g.density = ball_mass / (4.0 / 3.0 * math.pi * BALL_RADIUS**3)
            g.friction = [ball_friction, 0.005, 0.0001]

    goal = wb.add_body(name="goal", mocap=True, pos=[1, 0, 0])
    goal.add_geom(name="goal_marker", type=G.mjGEOM_CYLINDER, size=[0.25, 0.005, 0], rgba=[0.1, 0.9, 0.2, 0.6],
                  contype=0, conaffinity=0)
    return spec


mjm = build_ball_scene().compile()
print("nq =", mjm.nq, "nv =", mjm.nv, "| body:", [mjm.body(i).name for i in range(mjm.nbody)], "| 공 질량", mjm.body("ball/sphere").mass[0])


## Part 1 — `BatchedSim`: 환경 N 개를 GPU 에서 한꺼번에

섹터 0 에서는 `MjData` 하나를 `mj_step` 으로 돌렸습니다. 강화학습은 표본이 많이 필요해서, World 는 같은 모델의 상태 4096 벌을 GPU 에 올려 한 번에 스텝을 밟습니다. 그 일을 하는 라이브러리가 `mujoco_warp` (줄여서 `mjw`) 입니다.

```python
model = mjw.put_model(mjm)                      # CPU 의 MjModel → GPU 모델 (환경들이 공유)
data  = mjw.put_data(mjm, mjd, nworld=N)        # CPU 의 MjData 를 N 벌 복제한 GPU 데이터
mjw.step(model, data)                           # N 개 world 를 한 스텝 전진
mjw.forward(model, data)                        # mj_forward 의 배치판
```

`data.qpos` 는 모양이 `(N, nq)` 인 Warp 배열입니다. `wp.to_torch(data.qpos)` 를 부르면 **복사 없이** 같은 GPU 메모리를 가리키는 torch 텐서가 나옵니다. 이 텐서에 값을 쓰면 시뮬레이션 상태가 바뀝니다. 관측을 읽고, 리셋 때 상태를 써넣는 일을 전부 이 torch 뷰로 합니다.

각 world 는 서로 **완전히 독립** 입니다. world 0 의 공과 world 1 의 공은 같은 좌표에 있어도 부딪히지 않습니다. 그래서 환경을 격자로 띄워 놓을 필요가 없습니다(World 에는 `env_origins` 격자가 있지만 리셋 기준점으로만 씁니다).

### 문제 1-1. `BatchedSim`

만들 것:

- `__init__`: `put_model`, `put_data` 를 하고, 아래 torch 뷰를 속성으로 만든다.

| 속성 | 원본 | 모양 |
|---|---|---|
| `qpos`, `qvel` | `data.qpos`, `data.qvel` | `(N, nq)`, `(N, nv)` |
| `xpos`, `xquat` | `data.xpos`, `data.xquat` | `(N, nbody, 3)`, `(N, nbody, 4)` |
| `xfrc_applied` | `data.xfrc_applied` | `(N, nbody, 6)` |
| `mocap_pos` | `data.mocap_pos` | `(N, nmocap, 3)` |

- `step()`, `forward()`

`put_data` 에 넘길 `MjData` 는 `mj_forward` 를 한 번 해 둔 것이어야 `xpos` 등이 처음부터 맞습니다.


In [ ]:
class BatchedSim:
    """mujoco_warp 로 같은 모델의 world N 개를 GPU 에서 돌린다."""

    def __init__(self, mjm: mujoco.MjModel, num_envs: int, device: str = DEVICE):
        self.mjm = mjm
        self.mjd = mujoco.MjData(mjm)          # CPU 쪽 사본. 초기 상태 제공과 렌더링에 쓴다.
        mujoco.mj_forward(mjm, self.mjd)
        self.num_envs = num_envs
        self.device = device
        with wp.ScopedDevice(device):
            self.model = mjw.put_model(mjm)
            self.data = mjw.put_data(mjm, self.mjd, nworld=num_envs)
        self.qpos = wp.to_torch(self.data.qpos)
        self.qvel = wp.to_torch(self.data.qvel)
        self.xpos = wp.to_torch(self.data.xpos)
        self.xquat = wp.to_torch(self.data.xquat)
        self.xfrc_applied = wp.to_torch(self.data.xfrc_applied)
        self.mocap_pos = wp.to_torch(self.data.mocap_pos)

    @property
    def dt(self) -> float:
        return self.mjm.opt.timestep

    def step(self) -> None:
        mjw.step(self.model, self.data)

    def forward(self) -> None:
        mjw.forward(self.model, self.data)


In [ ]:
_t0 = time.time()
sim = BatchedSim(mjm, num_envs=8)
sim.step()
print(f"   (첫 스텝까지 {time.time() - _t0:.1f} 초 — 처음이면 커널 컴파일 시간 포함)")
check("qpos (N, 7), qvel (N, 6)", sim.qpos.shape == (8, 7) and sim.qvel.shape == (8, 6))
check("xpos (N, nbody, 3), xfrc_applied (N, nbody, 6)", sim.xpos.shape == (8, mjm.nbody, 3) and sim.xfrc_applied.shape == (8, mjm.nbody, 6))
check("mocap_pos (N, 1, 3)", sim.mocap_pos.shape == (8, 1, 3))
check("GPU 텐서", sim.qpos.is_cuda)
check("복사 없는 뷰: 같은 메모리를 가리킴", sim.qpos.data_ptr() == wp.to_torch(sim.data.qpos).data_ptr())

# 뷰에 쓰면 시뮬레이션 상태가 바뀐다: world 마다 다른 높이에서 떨어뜨리기
sim.qpos[:, 2] = torch.linspace(0.2, 1.6, 8, device=DEVICE)
sim.forward()
_bid = mjm.body("ball/sphere").id
check_close("forward 후 xpos 가 qpos 를 따라감", sim.xpos[:, _bid, 2], torch.linspace(0.2, 1.6, 8))
for _ in range(60):
    sim.step()
print("   0.3초 뒤 높이:", sim.qpos[:, 2])
check("world 마다 다른 궤적 (서로 독립)", sim.qpos[:, 2].unique().numel() == 8)


### 먼저 예상해 보기

world 4096 개를 100 스텝 돌리는 데(= 공 40만 스텝) 몇 초가 걸릴까요? 섹터 0 의 CPU `mj_step` 은 이 씬에서 초당 수십만 스텝쯤 나옵니다. world 를 8 개로 줄이면 100 스텝에 걸리는 시간은 512 분의 1 이 될까요?


In [ ]:
def _steps_per_second(num_envs, n_steps=100):
    s = BatchedSim(mjm, num_envs)
    s.step()                                   # 워밍업
    wp.synchronize()
    t = time.time()
    for _ in range(n_steps):
        s.step()
    wp.synchronize()                           # GPU 작업이 끝날 때까지 기다린 뒤 시간을 잰다
    return time.time() - t

for _n in (8, 512, 4096):
    _dt = _steps_per_second(_n)
    print(f"world {_n:5d} 개 × 100 스텝: {_dt * 1000:7.1f} ms   →  초당 {_n * 100 / _dt:12,.0f} 스텝")

_d = mujoco.MjData(mjm)
_t = time.time()
for _ in range(20000):
    mujoco.mj_step(mjm, _d)
print(f"CPU mj_step (world 1 개):                       →  초당 {20000 / (time.time() - _t):12,.0f} 스텝")


world 수를 512 배 늘려도 시간은 거의 늘지 않습니다. GPU 에서는 "한 스텝을 부르는 고정 비용" 이 크고 world 를 추가하는 비용은 작습니다. 그래서 환경 수는 GPU 메모리가 허락하는 만큼 키우고, 대신 **파이썬에서 world 를 하나씩 도는 반복문을 절대 쓰지 않는 것** 이 규칙입니다. 이 노트북의 모든 관측·보상·리셋 코드는 `(N, ...)` 텐서 연산으로 씁니다.

World 는 여기서 한 걸음 더 나가 물리 스텝 묶음을 CUDA 그래프로 캡처해 호출 비용까지 줄입니다(`diden_mjlab/env/mjwarp_env.py` 의 `perform_physics_step`).


## Part 2 — 쿼터니언 연산 (torch)

섹터 0 Part 2 에서 numpy 로 만든 `quat_rotate` 를 torch 로 다시 만듭니다. World 의 이름을 따라 `quat_apply`, `quat_apply_inverse` 로 부릅니다(`diden_common/math.py`). 쿼터니언은 `(w, x, y, z)` 입니다.

$$v' = v + 2w\,(\mathbf{u} \times v) + 2\,\mathbf{u} \times (\mathbf{u} \times v), \qquad q = (w, \mathbf{u})$$

### 문제 2-1


In [ ]:
def quat_apply(quat: torch.Tensor, vec: torch.Tensor) -> torch.Tensor:
    """body → world.  quat (..., 4) (w,x,y,z),  vec (..., 3)."""
    w, u = quat[..., :1], quat[..., 1:]
    t = torch.cross(u, vec, dim=-1)
    return vec + 2.0 * w * t + 2.0 * torch.cross(u, t, dim=-1)


def quat_apply_inverse(quat: torch.Tensor, vec: torch.Tensor) -> torch.Tensor:
    """world → body."""
    conj = torch.cat([quat[..., :1], -quat[..., 1:]], dim=-1)
    return quat_apply(conj, vec)


In [ ]:
_g = torch.Generator().manual_seed(0)
_q = torch.randn(64, 4, generator=_g, dtype=torch.float64); _q = _q / _q.norm(dim=-1, keepdim=True)
_v = torch.randn(64, 3, generator=_g, dtype=torch.float64)
_want = np.zeros((64, 3))
for _i in range(64):
    mujoco.mju_rotVecQuat(_want[_i], _v[_i].numpy(), _q[_i].numpy())
check_close("quat_apply == mju_rotVecQuat", quat_apply(_q, _v), _want, atol=1e-12)
check_close("quat_apply_inverse 가 되돌림", quat_apply_inverse(_q, quat_apply(_q, _v)), _v, atol=1e-12)
check("GPU, float32 에서도 동작", quat_apply(_q.to(DEVICE).float(), _v.to(DEVICE).float()).shape == (64, 3))


## Part 3 — `SimAdapter`: 엔진과 닿는 유일한 창구

관측·보상·리셋 코드가 `sim.qpos[:, 3:7]` 같은 엔진 내부 배열을 직접 만지면, 엔진을 바꿀 때 그 코드를 전부 다시 써야 합니다. World 는 MDP 코드가 엔진에 대해 알아야 하는 것을 `SimAdapter` 한 클래스에 모아 두고(`diden_common/sim_adapter.py`), 엔진마다 구현체를 둡니다(`diden_mjlab/sim_adapter.py` 의 `MujocoWarpSimAdapter`). 설계 문서는 이 경계를 **SEAM 2** 라고 부릅니다. `tests/guards/` 에는 "MDP 코드가 엔진을 직접 import 하지 않는가" 를 검사하는 테스트가 있습니다.

World 의 `SimAdapter` 는 포트 여섯 개(`RootStatePort`, `BodyQueryPort`, `JointControlPort`, `TerrainPort`, `SimWritePort`, `ContactPort`)로 나뉘어 있습니다. 공에는 관절도 지형도 없으므로 여기서는 **root 상태 읽기** 와 **쓰기** 만 만듭니다.

| 우리 메서드 | World 에서 |
|---|---|
| `root_pos_w`, `root_quat_w`, `root_lin_vel_w`, `root_ang_vel_w` | `RootStatePort` 의 `base_position`, `base_quat`, `base_lin_vel` … |
| `write_root_state(env_ids, ...)` | `SimWritePort` 의 root 쓰기 |
| `set_external_torque_w(torque)` | `SimWritePort.write_external_wrench_to_sim` (자석 힘도 이것으로 건다) |
| `geom_friction`, `write_geom_friction` | `SimWritePort.write_model_field_indexed` |

### 섹터 0 복습 두 가지

1. freejoint 의 `qvel` 은 앞 3칸(선속도)이 세계 좌표계, 뒤 3칸(각속도)이 **body 좌표계** 입니다.
2. `xfrc_applied[body] = [힘 3, 토크 3]` 은 세계 좌표계이고, 지울 때까지 유지됩니다.

### 환경마다 다른 모델 값

`put_model` 이 만든 GPU 모델에서 `model.geom_friction` 의 모양은 `(1, ngeom)` 입니다. 맨 앞 1 은 "모든 world 가 같은 값을 쓴다" 는 뜻입니다. 환경마다 마찰을 다르게 주려면 이 배열을 `(N, ngeom)` 로 **늘려서 다시 달아** 줘야 합니다. World 에서는 `MujocoWarpEnv.create_scene` 이 랜덤화할 필드들을 미리 늘려 둡니다(`diden_mjlab/env/mjwarp_env.py`).

```python
old = sim.model.geom_friction.numpy()                      # (1, ngeom, 3)
sim.model.geom_friction = wp.array(np.repeat(old, N, axis=0), dtype=wp.vec3, device=...)
```

### 문제 3-1. `SimAdapter`


In [ ]:
class SimAdapter:
    """MDP 코드가 시뮬레이터에 대해 아는 전부. 모든 텐서의 첫 축은 환경(world)이다."""

    def __init__(self, sim: BatchedSim, body_name: str = "ball/sphere", geom_name: str = "ball/sphere_collision"):
        self.sim = sim
        self.num_envs, self.device = sim.num_envs, sim.device
        mjm = sim.mjm
        self.body_id = mjm.body(body_name).id
        self.geom_id = mjm.geom(geom_name).id
        joint = mjm.joint(mjm.body_jntadr[self.body_id])           # body 의 첫 관절 = freejoint
        assert joint.type[0] == mujoco.mjtJoint.mjJNT_FREE
        self._qadr, self._vadr = int(joint.qposadr[0]), int(joint.dofadr[0])
        self.radius = float(mjm.geom(geom_name).size[0])
        self.mass = float(mjm.body(body_name).mass[0])
        old = sim.model.geom_friction.numpy()
        expanded = np.repeat(old, self.num_envs, axis=0) if old.shape[0] == 1 else old
        sim.model.geom_friction = wp.array(expanded, dtype=wp.vec3, device=self.device)
        self._geom_friction = wp.to_torch(sim.model.geom_friction)   # (N, ngeom, 3)

    # ── 읽기 ───────────────────────────────────────────────
    @property
    def root_pos_w(self) -> torch.Tensor:
        """(N, 3) 세계 좌표계 위치."""
        return self.sim.qpos[:, self._qadr:self._qadr + 3]

    @property
    def root_quat_w(self) -> torch.Tensor:
        """(N, 4) 자세 (w,x,y,z)."""
        return self.sim.qpos[:, self._qadr + 3:self._qadr + 7]

    @property
    def root_lin_vel_w(self) -> torch.Tensor:
        """(N, 3) 세계 좌표계 선속도."""
        return self.sim.qvel[:, self._vadr:self._vadr + 3]

    @property
    def root_ang_vel_w(self) -> torch.Tensor:
        """(N, 3) 세계 좌표계 각속도. (qvel 의 각속도는 body 좌표계!)"""
        return quat_apply(self.root_quat_w, self.sim.qvel[:, self._vadr + 3:self._vadr + 6])

    @property
    def geom_friction(self) -> torch.Tensor:
        """(N,) 공 collision geom 의 미끄럼 마찰계수."""
        return self._geom_friction[:, self.geom_id, 0]

    # ── 쓰기 ───────────────────────────────────────────────
    def write_root_state(self, env_ids: torch.Tensor, pos: torch.Tensor, quat: torch.Tensor,
                         lin_vel_w: torch.Tensor, ang_vel_w: torch.Tensor) -> None:
        """env_ids 환경의 공 상태를 덮어쓴다. 속도는 둘 다 세계 좌표계로 받는다."""
        self.sim.qpos[env_ids, self._qadr:self._qadr + 3] = pos
        self.sim.qpos[env_ids, self._qadr + 3:self._qadr + 7] = quat
        self.sim.qvel[env_ids, self._vadr:self._vadr + 3] = lin_vel_w
        self.sim.qvel[env_ids, self._vadr + 3:self._vadr + 6] = quat_apply_inverse(quat, ang_vel_w)

    def set_external_torque_w(self, torque_w: torch.Tensor) -> None:
        """(N, 3) 세계 좌표계 토크를 공에 건다. 다음에 다시 부를 때까지 유지된다."""
        self.sim.xfrc_applied[:, self.body_id, 3:6] = torque_w

    def write_geom_friction(self, env_ids: torch.Tensor, friction: torch.Tensor) -> None:
        """env_ids 환경의 공 미끄럼 마찰계수를 바꾼다."""
        self._geom_friction[env_ids, self.geom_id, 0] = friction

    def set_goal_marker(self, goal_pos_w: torch.Tensor) -> None:
        """(주어진 코드) 목적지 표시(mocap body)를 옮긴다. 물리에는 영향이 없고 그림에만 쓰인다."""
        self.sim.mocap_pos[:, 0, :] = goal_pos_w


In [ ]:
N = 4096
sim = BatchedSim(mjm, N)
adapter = SimAdapter(sim)
check("읽기 모양", adapter.root_pos_w.shape == (N, 3) and adapter.root_quat_w.shape == (N, 4)
      and adapter.root_lin_vel_w.shape == (N, 3) and adapter.root_ang_vel_w.shape == (N, 3))
check("마찰이 환경마다 따로 (N, ngeom, 3)", tuple(sim.model.geom_friction.shape) == (N, mjm.ngeom) and adapter.geom_friction.shape == (N,))

# 쓰기 → 읽기 왕복
_ids = torch.tensor([0, 7, 4095], device=DEVICE)
_quat = torch.tensor([[math.cos(math.pi / 4), 0.0, 0.0, math.sin(math.pi / 4)]], device=DEVICE).expand(3, 4)   # z 축 90°
_ang_w = torch.tensor([[0.0, 1.0, 0.0]], device=DEVICE).expand(3, 3)
adapter.write_root_state(_ids, torch.tensor([[1.0, 2.0, 0.2]], device=DEVICE).expand(3, 3), _quat,
                         torch.tensor([[0.3, 0.0, 0.0]], device=DEVICE).expand(3, 3), _ang_w)
sim.forward()
check_close("위치 왕복", adapter.root_pos_w[_ids], [[1.0, 2.0, 0.2]] * 3)
check_close("세계 각속도 왕복", adapter.root_ang_vel_w[_ids], _ang_w, atol=1e-6)
check_close("qvel 에는 body 좌표계로 들어감: 공이 z 로 90° 돌아 있으므로 world y = body x", sim.qvel[_ids, 3:6], [[1.0, 0.0, 0.0]] * 3, atol=1e-6)
check_close("다른 환경은 그대로", adapter.root_pos_w[1], [0.0, 0.0, 0.2], atol=1e-6)


### 4096 개 환경으로 실험 한 번에 하기

섹터 0 문제 4-5 에서 "토크를 키우면 어느 순간 헛돈다" 를 토크 두 개로 확인했습니다. 이제 환경이 4096 개 있으니, **토크를 0 에서 4 N·m 까지 4096 단계로** 나눠 한 번에 돌릴 수 있습니다. 마찰도 절반은 0.3, 절반은 1.0 으로 줍니다.

### 먼저 예상해 보기

가로축 토크, 세로축 1초 뒤 속도(= 평균 가속도) 그래프는 어떤 모양일까요? 마찰이 0.3 인 환경과 1.0 인 환경의 그래프는 어디서 갈라질까요? ($a = \tau / (\tfrac75 m r)$, 헛돌 때 $a \approx \mu g$)


In [ ]:
sim = BatchedSim(mjm, N)
adapter = SimAdapter(sim)
_all = torch.arange(N, device=DEVICE)
_mu = torch.where(_all % 2 == 0, 0.3, 1.0)
adapter.write_geom_friction(_all, _mu)
_tau = torch.linspace(0.0, 4.0, N, device=DEVICE)
_torque = torch.zeros(N, 3, device=DEVICE)
_torque[:, 1] = _tau                                    # 세계 y 축 토크 → +x 로 굴러간다
adapter.set_external_torque_w(_torque)
for _ in range(200):                                    # 1초
    sim.step()
_accel = adapter.root_lin_vel_w[:, 0] / (200 * sim.dt)
_slip = adapter.root_lin_vel_w[:, 0] - adapter.root_ang_vel_w[:, 1] * adapter.radius      # 접촉점 미끄럼 속도

plt.figure(figsize=(7, 3.6))
for _m, _c in ((0.3, "tab:orange"), (1.0, "tab:blue")):
    _sel = _mu == _m
    plt.plot(_np(_tau[_sel]), _np(_accel[_sel]), color=_c, label=f"sim, mu = {_m}")
    plt.axhline(_m * GRAVITY, color=_c, ls=":", lw=1)
    plt.axvline(1.4 * _m * BALL_MASS * GRAVITY * BALL_RADIUS, color=_c, ls=":", lw=1)
plt.plot(_np(_tau), _np(_tau) / (1.4 * BALL_MASS * BALL_RADIUS), "k--", lw=1, label="no slip: tau / (7/5 m r)")
plt.xlabel("torque [N m]"); plt.ylabel("acceleration [m/s^2]"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
_noslip = (_mu == 1.0) & (_tau > 0.2) & (_tau < 0.8 * 1.4 * 1.0 * BALL_MASS * GRAVITY * BALL_RADIUS)
check_close("미끄러지지 않는 구간: a = τ / (7/5·m·r)", _accel[_noslip], _tau[_noslip] / (1.4 * BALL_MASS * BALL_RADIUS), rtol=0.02)
check("그 구간에서는 접촉점이 거의 미끄러지지 않음", _slip[_noslip].abs().max() < 0.05)
_slipping = (_mu == 0.3) & (_tau > 2.0)
check("μ=0.3, τ>2: 헛돈다 (접촉점 미끄럼 속도 < −1 m/s)", (_slip[_slipping] < -1.0).all())
check("헛도는 구간: 평균 가속도가 μg 근처 (±20%)", abs(_accel[_slipping].mean().item() / (0.3 * GRAVITY) - 1) < 0.2)
check("헛도는 구간: 토크를 더 줘도 가속도는 '미끄러지지 않을 때' 값에 한참 못 미침", (_accel[_slipping] < 0.75 * _tau[_slipping] / (1.4 * BALL_MASS * BALL_RADIUS)).all())
print(f"   μ=0.3 에서 헛돌 때 가속도: {_accel[_slipping].mean():.2f} m/s²  (μg = {0.3 * GRAVITY:.2f})")


헛도는 구간의 그래프는 $\mu g$ 주변에서 들쭉날쭉합니다. 섹터 0 에서 본 "부드러운 접촉" 때문입니다. 미끄러지는 동안의 마찰력은 제약 풀이(여기서는 반복 10회)의 결과라서 쿨롱 마찰 $\mu N$ 과 수십 % 까지 차이가 납니다. 시뮬레이터의 마찰은 "미끄러지지 않는 구간" 에서는 정확하고 "미끄러지는 구간" 에서는 근사라는 점을 기억해 둡니다.

이 그래프가 Part 6 에서 **액션 스케일** 을 정하는 근거입니다. 마찰을 0.3~1.0 으로 랜덤화하면 헛돌지 않는 최대 토크가 환경마다 0.82~2.75 N·m 로 달라집니다. 정책은 마찰을 직접 볼 수 없으므로(actor 관측에 넣지 않습니다), 속도 변화를 보고 스스로 조절하는 법을 배워야 합니다.

**실제 코드에서는**: World 의 `SimAdapter` 는 읽은 값을 스텝마다 **캐시** 합니다(`diden_common/sim_state_cache.py`). 보상 항 30 개가 각자 `base_lin_vel` 을 부르면 body 좌표계 변환을 30 번 하게 되기 때문입니다. 캐시는 물리 서브스텝마다 무효화됩니다(`ManagerBasedRlEnv.on_physics_substep`).


## Part 4 — 설정과 term: 매니저 방식의 뼈대

World 에서 환경 하나를 정의한다는 것은 **설정 클래스 하나를 쓰는 것** 입니다. 관측 항, 보상 항, 종료 조건을 각각 "함수 + 인자" 로 적어 두면 매니저가 그것을 읽어 매 스텝 호출합니다. Spider 보행의 MDP 정의 전체가 `diden_configs/robots/diden_spider/magnet_locomotion_mdp.py` 한 파일에 이런 모양으로 들어 있습니다.

```python
@configclass
class RewardsCfg:
    tracking_lin_vel = RewTerm(func=mdp.track_lin_vel_xy_exp, weight=100.0, params={"std": 0.5})
    foot_slip        = RewTerm(func=mdp.foot_slip, weight=800.0, operation=Operation.multiply, transformation=small_exp)
    ...
```

이 방식의 장점은 "무엇을 계산하는가"(MDP 함수)와 "언제·어떻게 모으는가"(매니저)가 분리된다는 것입니다. 보상 항을 하나 추가하는 일이 설정 한 줄이 됩니다.

용어:

- **term**: 관측 항 하나, 보상 항 하나 같은 최소 단위. 설정은 `...TermCfg`.
- **func**: term 을 계산하는 함수. 첫 인자는 항상 `env` 이고, `(N, ...)` 텐서를 돌려줍니다.
- **params**: func 에 넘길 나머지 인자.
- **manager**: 같은 종류의 term 들을 순서대로 불러 결과를 합치는 객체.

### `configclass`

World 의 설정 클래스는 `@configclass` 로 꾸며져 있습니다(`diden_common/configclass.py`). 파이썬 `dataclass` 에 두 가지를 더한 것입니다. (1) 타입 표기 없이 `name = 값` 으로만 써도 필드가 되고, (2) 기본값이 객체여도 인스턴스마다 따로 복사됩니다. 아래는 그 축소판이고 주어진 코드입니다.


In [ ]:
import copy
import dataclasses
import inspect


def configclass(cls):
    """dataclass + (타입 표기 없는 속성도 필드로) + (객체 기본값은 인스턴스마다 깊은 복사)."""
    ann = dict(cls.__dict__.get("__annotations__", {}))
    for name, value in list(cls.__dict__.items()):
        if name.startswith("_") or name in ann:
            continue
        if inspect.isfunction(value) or isinstance(value, (type, property, classmethod, staticmethod)):
            continue
        ann[name] = type(value)
    cls.__annotations__ = ann
    for name in ann:
        if name in cls.__dict__:
            value = cls.__dict__[name]
            if isinstance(value, dataclasses.Field):
                continue
            if not isinstance(value, (int, float, str, bool, tuple, type(None))) and not callable(value):
                setattr(cls, name, field(default_factory=lambda v=value: copy.deepcopy(v)))
    return dataclass(cls)


@configclass
class ManagerTermBaseCfg:
    """모든 term 설정의 부모. (World: diden_common/manager_term_cfg.py)"""
    func: Callable = None
    params: dict = field(default_factory=dict)


class ManagerTermBase:
    """상태를 가져야 하는 term 은 함수 대신 이 클래스를 상속해 만든다. (World: manager_term_base.py)"""

    def __init__(self, cfg, env):
        self.cfg, self._env = cfg, env

    @property
    def num_envs(self) -> int:
        return self._env.num_envs

    @property
    def device(self) -> str:
        return self._env.device

    def reset(self, env_ids=None) -> None:
        pass

    def __call__(self, *args, **kwargs):
        raise NotImplementedError


### 문제 4-1. 설정에서 term 꺼내기

매니저는 생성될 때 설정 객체의 필드를 **정의된 순서대로** 훑어 term 을 모읍니다(World: 각 매니저의 `_prepare_terms`). 순서가 중요한 이유는 관측 벡터의 칸 배치가 이 순서로 정해지기 때문입니다. 학습 때와 배포 때 순서가 다르면 정책은 엉뚱한 값을 읽습니다.

`iter_terms(cfg)` 는 `(이름, term 설정)` 목록을 돌려줍니다.

- `dataclasses.fields(cfg)` 가 필드를 정의 순서대로 줍니다.
- 값이 `None` 인 필드는 건너뜁니다(World 에서 term 을 끄는 방법이 `term = None` 입니다).
- `ManagerTermBaseCfg` 의 인스턴스가 아닌 필드(예: 그룹 옵션 `enable_corruption`)도 건너뜁니다.

### 문제 4-2. func 준비하기

term 의 `func` 은 함수일 수도 있고 `ManagerTermBase` 를 상속한 **클래스** 일 수도 있습니다. 클래스면 `func(cfg=term_cfg, env=env)` 로 인스턴스를 만들어 그것을 호출 가능한 객체로 씁니다(World: `ManagerBase._resolve_common_term_cfg`). `resolve_term_func` 은 어느 쪽이든 "부를 수 있는 것" 을 돌려줍니다.


In [ ]:
def iter_terms(cfg) -> list[tuple[str, Any]]:
    """설정 객체에서 (이름, term 설정) 을 정의 순서대로 뽑는다."""
    out = []
    for f in fields(cfg):
        value = getattr(cfg, f.name)
        if value is None or not isinstance(value, ManagerTermBaseCfg):
            continue
        out.append((f.name, value))
    return out


def resolve_term_func(term_cfg, env) -> Callable:
    """term_cfg.func 이 클래스면 인스턴스를 만들어 돌려주고, 함수면 그대로 돌려준다."""
    if inspect.isclass(term_cfg.func):
        return term_cfg.func(cfg=term_cfg, env=env)
    return term_cfg.func


In [ ]:
def _f1(env): return 1
def _f2(env, k): return k

class _Counter(ManagerTermBase):
    def __init__(self, cfg, env):
        super().__init__(cfg, env)
        self.n = 0
    def __call__(self, env):
        self.n += 1
        return self.n

@configclass
class _DemoCfg:
    some_option: bool = True
    b_term = ManagerTermBaseCfg(func=_f2, params={"k": 7})
    a_term = ManagerTermBaseCfg(func=_f1)
    off_term: ManagerTermBaseCfg = None
    c_term = ManagerTermBaseCfg(func=_Counter)

_cfg = _DemoCfg()
_terms = iter_terms(_cfg)
check("정의 순서 유지 (이름순이 아님), None 과 옵션은 제외", [n for n, _ in _terms] == ["b_term", "a_term", "c_term"])
check("인스턴스마다 설정이 따로 (깊은 복사)", _DemoCfg().b_term is not _cfg.b_term)
_fn = resolve_term_func(_terms[0][1], env=None)
check("함수 term", _fn(None, **_terms[0][1].params) == 7)
_cls_fn = resolve_term_func(_terms[2][1], env=None)
check("클래스 term 은 인스턴스가 되어 상태를 가진다", isinstance(_cls_fn, _Counter) and _cls_fn(None) == 1 and _cls_fn(None) == 2)


## Part 5 — `CommandManager`: 목적지

**명령(command)** 은 "정책에게 시키는 일" 입니다. Spider 보행에서는 원하는 속도 $(v_x, v_y, \omega_z)$ 이고, 여기서는 목적지의 위치 $(x, y)$ 입니다. 명령은 환경이 임의로 뽑고, 일정 시간이 지나면 다시 뽑습니다(resample). 한 에피소드 안에서 명령이 바뀌어야 정책이 "가는 법" 뿐 아니라 "방향을 바꾸는 법" 도 배웁니다.

World 의 구조(`diden_env/managers/command_manager.py`)를 그대로 따릅니다.

- `CommandTerm`: 명령 하나. 환경마다 남은 시간 `time_left` 를 들고 있다가 0 이하가 되면 그 환경만 다시 뽑습니다.
- `CommandManager`: term 들을 이름으로 보관하고 `compute(dt)`, `reset(env_ids)`, `get_command(name)` 을 제공합니다.

Spider 의 설정은 `resampling_time_range=(3, 6)` 초입니다. 우리는 `(4, 6)` 초, 에피소드 길이 8 초로 둡니다. 한 에피소드에 목적지가 한두 번 바뀝니다.

### 문제 5-1. 균등분포에서 뽑기

`(lo, hi)` 범위에서 `shape` 모양으로 뽑는 도우미입니다. `torch.rand` 는 [0, 1) 에서 뽑습니다.


In [ ]:
def sample_uniform(lo: float, hi: float, shape, device=DEVICE) -> torch.Tensor:
    return lo + (hi - lo) * torch.rand(shape, device=device)


In [ ]:
torch.manual_seed(0)
_s = sample_uniform(-3.0, 5.0, (100000, 2))
check("모양과 범위", _s.shape == (100000, 2) and _s.min() >= -3.0 and _s.max() < 5.0)
check_close("평균 ≈ 1, 표준편차 ≈ 8/√12", [_s.mean().item(), _s.std().item()], [1.0, 8 / math.sqrt(12)], atol=0.03)


### 문제 5-2. `CommandTerm` 과 목적지 명령

`CommandTerm` 의 두 메서드를 채웁니다.

- `_resample(env_ids)`: 그 환경들의 `time_left` 를 `resampling_time_range` 에서 새로 뽑고, `_resample_command(env_ids)` 를 부른다.
- `compute(dt)`: 모든 환경의 `time_left` 를 `dt` 만큼 줄이고, 0 이하가 된 환경만 `_resample` 한다.

`UniformGoalCommand._resample_command(env_ids)` 는 그 환경들의 목적지 xy 를 범위 안에서 새로 뽑아 `self.goal_pos_w` 에 넣습니다.

<details><summary>힌트</summary>

조건을 만족하는 환경 번호는 `(self.time_left <= 0.0).nonzero(as_tuple=False).squeeze(-1)` 로 얻습니다. 하나도 없으면 길이 0 텐서가 나오므로 `if len(env_ids) > 0:` 으로 건너뜁니다.
</details>


In [ ]:
@configclass
class CommandTermCfg:
    class_type: type = None
    resampling_time_range: tuple = (4.0, 6.0)


class CommandTerm:
    """명령 하나. (World: command_manager.py 의 CommandTerm)"""

    def __init__(self, cfg: CommandTermCfg, env):
        self.cfg, self._env = cfg, env
        self.num_envs, self.device = env.num_envs, env.device
        self.time_left = torch.zeros(self.num_envs, device=self.device)
        self.command_counter = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)   # 에피소드 안에서 몇 번째 명령인가

    @property
    def command(self) -> torch.Tensor:
        raise NotImplementedError

    def _resample_command(self, env_ids: torch.Tensor) -> None:
        raise NotImplementedError

    def _resample(self, env_ids: torch.Tensor) -> None:
        lo, hi = self.cfg.resampling_time_range
        self.time_left[env_ids] = sample_uniform(lo, hi, (len(env_ids),), self.device)
        self._resample_command(env_ids)
        self.command_counter[env_ids] += 1

    def compute(self, dt: float) -> None:
        self.time_left -= dt
        env_ids = (self.time_left <= 0.0).nonzero(as_tuple=False).squeeze(-1)
        if len(env_ids) > 0:
            self._resample(env_ids)

    def reset(self, env_ids: torch.Tensor) -> None:
        self.command_counter[env_ids] = 0
        self._resample(env_ids)


class UniformGoalCommand(CommandTerm):
    """세계 좌표계의 목적지 (x, y)."""

    def __init__(self, cfg: "UniformGoalCommandCfg", env):
        super().__init__(cfg, env)
        self.goal_pos_w = torch.zeros(self.num_envs, 2, device=self.device)

    @property
    def command(self) -> torch.Tensor:
        return self.goal_pos_w

    def _resample_command(self, env_ids: torch.Tensor) -> None:
        n = len(env_ids)
        self.goal_pos_w[env_ids, 0] = sample_uniform(*self.cfg.pos_x, (n,), self.device)
        self.goal_pos_w[env_ids, 1] = sample_uniform(*self.cfg.pos_y, (n,), self.device)
        # 그림용: 목적지 표시를 옮긴다 (World 에서는 debug_vis 가 하는 일)
        marker = torch.zeros(self.num_envs, 3, device=self.device)
        marker[:, :2] = self.goal_pos_w
        self._env.sim_adapter.set_goal_marker(marker)


@configclass
class UniformGoalCommandCfg(CommandTermCfg):
    class_type: type = UniformGoalCommand
    pos_x: tuple = (-3.0, 3.0)
    pos_y: tuple = (-3.0, 3.0)


class CommandManager:
    """(주어진 코드) 명령 term 들을 이름으로 보관한다."""

    def __init__(self, cfg, env):
        self._env = env
        self._terms = {f.name: getattr(cfg, f.name).class_type(getattr(cfg, f.name), env)
                       for f in fields(cfg) if getattr(cfg, f.name) is not None}

    def get_command(self, name: str) -> torch.Tensor:
        return self._terms[name].command

    def get_term(self, name: str) -> CommandTerm:
        return self._terms[name]

    def compute(self, dt: float) -> None:
        for term in self._terms.values():
            term.compute(dt)

    def reset(self, env_ids: torch.Tensor) -> dict:
        for term in self._terms.values():
            term.reset(env_ids)
        return {}


In [ ]:
class _FakeEnv:
    """매니저를 하나씩 따로 시험하기 위한 가짜 환경. 진짜 환경은 Part 11 에서 만든다."""

    def __init__(self, num_envs=4096):
        self.num_envs, self.device = num_envs, DEVICE
        self.sim = BatchedSim(mjm, num_envs)
        self.sim_adapter = SimAdapter(self.sim)
        self.step_dt = 0.02
        self.max_episode_length = 400
        self.max_episode_length_s = 8.0
        self.episode_length_buf = torch.zeros(num_envs, dtype=torch.long, device=DEVICE)

@configclass
class CommandsCfg:
    goal = UniformGoalCommandCfg(resampling_time_range=(4.0, 6.0), pos_x=(-3.0, 3.0), pos_y=(-3.0, 3.0))

torch.manual_seed(0)
fake = _FakeEnv()
fake.command_manager = CommandManager(CommandsCfg(), fake)
_term = fake.command_manager.get_term("goal")
_all = torch.arange(fake.num_envs, device=DEVICE)
fake.command_manager.reset(_all)
_g0 = fake.command_manager.get_command("goal").clone()
check("명령 모양 (N, 2), 범위 ±3", _g0.shape == (4096, 2) and _g0.abs().max() <= 3.0)
check("환경마다 다른 목적지", _g0.unique(dim=0).shape[0] > 4000)
check("남은 시간이 4~6 초", _term.time_left.min() >= 4.0 and _term.time_left.max() < 6.0)
for _ in range(199):                       # 3.98 초: 아직 아무도 만료되지 않는다
    fake.command_manager.compute(0.02)
check_close("만료 전에는 명령이 그대로", fake.command_manager.get_command("goal"), _g0, atol=0)
for _ in range(60):                        # 5.18 초 시점
    fake.command_manager.compute(0.02)
_changed = (fake.command_manager.get_command("goal") != _g0).any(dim=1).float().mean().item()
print(f"   5.18초 시점에 목적지가 바뀐 환경 비율: {_changed:.3f}  (기대값 (5.18−4)/2 = 0.59)")
check("일부 환경만 다시 뽑힘 (0.5 ~ 0.7)", 0.5 < _changed < 0.7)
check("다시 뽑힌 환경의 command_counter 는 2", (_term.command_counter == 2).float().mean().item() == _changed)
check_close("목적지 표시(mocap)가 명령을 따라감", fake.sim.mocap_pos[:, 0, :2], fake.command_manager.get_command("goal"), atol=0)


**실제 코드에서는**: Spider 의 명령 term 은 `UniformValidPosturalVelocityCommand` 입니다(`diden_env/mdp/commands/velocity_command.py`). 속도를 그냥 균등하게 뽑지 않고 55 % 는 직진만, 40 % 는 회전만 하도록 섞습니다. 명령 분포를 어떻게 잡는가가 정책이 무엇을 잘하게 되는가를 정합니다.


## Part 6 — `ActionManager`: 정책 출력에서 토크까지

정책 네트워크가 내는 숫자는 단위가 없습니다. 대략 −1 ~ 1 근처의 값이고, 학습 초반에는 평균 0, 표준편차 1 쯤의 잡음입니다. 이것을 물리량으로 바꾸는 것이 action term 입니다.

Spider 의 관절 액션(`diden_env/mdp/actions/joint_actions.py`)은

$$q^{*} = \text{raw} \times \text{scale} + \text{offset}$$

이고(`scale = 0.2 rad`, `offset = DEFAULT_JOINT_ANGLES_XML`), 이 $q^*$ 가 PD 제어기의 목표가 됩니다. 섹터 0 Part 6 에서 본 그 PD 입니다. Core 는 같은 식을 `action.cpp` 에서 `q = action_mean + joint_action_scale * raw` 로 다시 계산합니다. **scale 과 offset 이 두 레포에서 한 글자라도 다르면 sim-to-real 이 깨지므로**, World 는 이 값들을 ONNX 파일 안에 메타데이터로 같이 넣어 보냅니다(Part 17).

공에는 관절이 없으니 우리 액션은 토크입니다. 정책 출력 2개 $(a_x, a_y)$ 를 "그 방향으로 굴러가게 하는 토크" 로 바꿉니다.

$$\tau^{W} = s \cdot (\hat z \times \mathbf{a}) = s \cdot (-a_y,\; a_x,\; 0)$$

$a = (1, 0)$ 이면 +y 축 토크가 걸려 공이 +x 로 굴러갑니다(섹터 0 문제 4-5).

**scale 정하기.** Part 3 의 그래프에서, 헛돌지 않는 최대 토크는 마찰 0.3 일 때 0.82, 1.0 일 때 2.75 N·m 였습니다. `scale = 1.0 N·m`, `clip = 2.0` 으로 둡니다. 정책 출력 1 이 1 N·m (가속도 약 3.6 m/s²)이고, 최대 2 N·m 까지 낼 수 있습니다. 마찰이 낮은 환경에서는 최대 토크를 쓰면 헛돕니다.

### 한 스텝 안의 두 박자

`ActionManager` 에는 메서드가 두 개 있고 불리는 빈도가 다릅니다.

- `process_action(action)`: 정책 스텝마다 **한 번**. raw 를 저장하고 clip·scale 을 계산해 둔다.
- `apply_action()`: 물리 서브스텝마다 **매번** (`decimation` 번). 계산해 둔 값을 시뮬레이터에 써넣는다.

Spider 에서는 정책이 100 Hz, 물리가 1000 Hz 로 돌아서 `decimation = 10` 입니다. 우리는 정책 50 Hz, 물리 200 Hz, `decimation = 4` 입니다.

### 문제 6-1. `BallTorqueAction` 과 `ActionManager`

`ActionManager` 는 `action`(이번 스텝의 raw)과 `prev_action`(직전 스텝의 raw)을 보관합니다. 둘 다 관측과 보상에서 씁니다. 리셋된 환경은 둘 다 0 으로 만듭니다.


In [ ]:
@configclass
class ActionTermCfg:
    class_type: type = None
    scale: float = 1.0
    clip: float = None            # raw 액션을 ±clip 으로 자른다 (None 이면 자르지 않음)


class ActionTerm:
    action_dim: int = 0

    def __init__(self, cfg: ActionTermCfg, env):
        self.cfg, self._env = cfg, env
        self.num_envs, self.device = env.num_envs, env.device
        self.raw_actions = torch.zeros(self.num_envs, self.action_dim, device=self.device)
        self.processed_actions = torch.zeros_like(self.raw_actions)

    def process_actions(self, actions: torch.Tensor) -> None:
        raise NotImplementedError

    def apply_actions(self) -> None:
        raise NotImplementedError

    def reset(self, env_ids: torch.Tensor) -> None:
        self.raw_actions[env_ids] = 0.0
        self.processed_actions[env_ids] = 0.0


class BallTorqueAction(ActionTerm):
    """정책 출력 (a_x, a_y) → 세계 좌표계 토크 s·(−a_y, a_x, 0)."""
    action_dim = 2

    def process_actions(self, actions: torch.Tensor) -> None:
        self.raw_actions[:] = actions
        a = actions if self.cfg.clip is None else actions.clamp(-self.cfg.clip, self.cfg.clip)
        self.processed_actions[:] = a * self.cfg.scale

    def apply_actions(self) -> None:
        p = self.processed_actions
        torque = torch.stack([-p[:, 1], p[:, 0], torch.zeros_like(p[:, 0])], dim=-1)
        self._env.sim_adapter.set_external_torque_w(torque)


@configclass
class BallTorqueActionCfg(ActionTermCfg):
    class_type: type = BallTorqueAction


class ActionManager:
    """(World: diden_env/managers/action_manager.py)"""

    def __init__(self, cfg, env):
        self._env = env
        self._terms = {f.name: getattr(cfg, f.name).class_type(getattr(cfg, f.name), env)
                       for f in fields(cfg) if getattr(cfg, f.name) is not None}
        self.total_action_dim = sum(t.action_dim for t in self._terms.values())
        self.action = torch.zeros(env.num_envs, self.total_action_dim, device=env.device)
        self.prev_action = torch.zeros_like(self.action)

    def process_action(self, action: torch.Tensor) -> None:
        self.prev_action[:] = self.action
        self.action[:] = action
        idx = 0
        for term in self._terms.values():
            term.process_actions(action[:, idx:idx + term.action_dim])
            idx += term.action_dim

    def apply_action(self) -> None:
        for term in self._terms.values():
            term.apply_actions()

    def reset(self, env_ids: torch.Tensor) -> dict:
        self.action[env_ids] = 0.0
        self.prev_action[env_ids] = 0.0
        for term in self._terms.values():
            term.reset(env_ids)
        return {}


In [ ]:
@configclass
class ActionsCfg:
    torque = BallTorqueActionCfg(scale=1.0, clip=2.0)

fake = _FakeEnv(num_envs=8)
fake.action_manager = ActionManager(ActionsCfg(), fake)
check("액션 차원 2", fake.action_manager.total_action_dim == 2)
_a = torch.zeros(8, 2, device=DEVICE)
_a[:, 0] = torch.tensor([0.0, 0.5, 1.0, 2.0, 5.0, -1.0, 0.0, 0.0])
_a[6, 1] = 1.0
fake.action_manager.process_action(_a)
_term = fake.action_manager._terms["torque"]
check_close("raw 는 그대로 보관", _term.raw_actions, _a)
check_close("processed 는 clip(±2) 후 scale", _term.processed_actions[:, 0], [0.0, 0.5, 1.0, 2.0, 2.0, -1.0, 0.0, 0.0])
check("process 만으로는 시뮬레이터에 아무것도 쓰지 않음", fake.sim.xfrc_applied.abs().sum() == 0)
fake.action_manager.apply_action()
_bid = fake.sim_adapter.body_id
check_close("a=(1,0) → +y 토크", fake.sim.xfrc_applied[2, _bid, 3:6], [0.0, 1.0, 0.0])
check_close("a=(0,1) → −x 토크", fake.sim.xfrc_applied[6, _bid, 3:6], [-1.0, 0.0, 0.0])
for _ in range(100):                                   # 0.5 초
    fake.action_manager.apply_action()
    fake.sim.step()
_v = fake.sim_adapter.root_lin_vel_w
check_close("a=(1,0): +x 로 가속, a = τ/(7/5·m·r)", _v[2, 0] / 0.5, 1.0 / (1.4 * BALL_MASS * BALL_RADIUS), rtol=0.02)
check_close("a=(0,1): +y 로 가속", _v[6, 1] / 0.5, 1.0 / (1.4 * BALL_MASS * BALL_RADIUS), rtol=0.02)
check("a=(−1,0): −x 로", _v[5, 0] < -1.0)
fake.action_manager.process_action(torch.ones(8, 2, device=DEVICE))
check_close("prev_action 은 직전 스텝의 raw", fake.action_manager.prev_action, _a)
fake.action_manager.reset(torch.tensor([0, 1], device=DEVICE))
check("리셋된 환경만 0", fake.action_manager.action[:2].abs().sum() == 0 and fake.action_manager.prev_action[:2].abs().sum() == 0
      and fake.action_manager.action[2:].abs().sum() > 0)


**실제 코드에서는**

- **액션 지연.** 실제 로봇은 관측을 읽고 → 추론하고 → 명령이 모터에 도착하기까지 몇 ms 가 걸립니다. World 는 에피소드마다 3~5 ms, 스텝마다 ±2 ms 의 지연을 뽑아, `apply_action` 을 서브스텝 0 이 아니라 그만큼 뒤의 서브스텝에서 적용합니다(`diden_mjlab/env/mjwarp_env.py` 의 `perform_physics_step`, sim profile 의 action latency). 물리를 정책보다 잘게 쪼개 도는 이유 중 하나가 이 지연을 ms 단위로 흉내 내기 위해서입니다.
- **clip 을 하지 않습니다.** Spider 의 정책은 `clip_actions=False` 입니다. 대신 보상에서 큰 액션과 급한 변화를 벌합니다.
- **자석 액션.** Spider 에는 관절 12 개 외에 발 자석 4 개의 on/off 액션이 있고(`diden_domain/mdp/actions/magnet_actions.py`), `tanh` 를 거쳐 상태기계로 들어갑니다. `ActionManager` 가 액션 벡터를 term 별로 **잘라서** 나눠 주는 이유입니다(위의 `idx` 반복문).


## Part 7 — `ObservationManager`: 정책이 보는 것

관측을 설계할 때 묻는 질문은 세 가지입니다.

1. **실제 로봇에서 잴 수 있는가?** 잴 수 없는 값을 정책에 주면 시뮬레이션에서만 잘 됩니다.
2. **과제를 푸는 데 충분한가?** 목적지가 어디인지 모르면 갈 수 없습니다.
3. **좌표계와 크기가 학습하기 좋은가?**

World 는 관측을 **그룹** 으로 나눕니다(`magnet_locomotion_mdp.py` 의 `ObservationsCfg`).

- `actor`: 정책 네트워크의 입력. 실제 센서로 얻을 수 있는 값만, **노이즈를 섞어서** 줍니다.
- `critic`: 가치 네트워크의 입력. 학습 때만 쓰이고 로봇에는 올라가지 않으므로, 시뮬레이터만 아는 "정답" 을 **노이즈 없이** 다 줍니다(privileged observation). Spider 의 critic 은 actor 항들의 깨끗한 버전에 몸통 선속도, 발 접촉 여부, 발 높이, 자석 힘 상태를 더한 것입니다.

우리 환경의 관측:

| 항 | 차원 | actor | critic | 실제라면 어떻게 잴까 |
|---|---|---|---|---|
| `ball_lin_vel` — 공의 선속도 xy (세계) | 2 | 노이즈 ±0.05 | 깨끗 | 외부 카메라, 또는 추정 |
| `ball_ang_vel` — 공의 각속도 (세계) | 3 | 노이즈 ±0.2 | 깨끗 | 자이로 |
| `goal_rel_pos` — 목적지 − 공 위치 xy | 2 | 노이즈 ±0.02 | 깨끗 | 카메라 |
| `last_action` — 직전 액션 | 2 | 그대로 | 그대로 | 내가 낸 값이므로 정확히 안다 |
| `ball_friction` — 마찰계수 | 1 | **없음** | 있음 | 잴 수 없다 |

합계: actor 9, critic 10.

**왜 목적지의 절대 좌표가 아니라 상대 위치인가?** 공이 (0,0) 에서 (1,0) 으로 가는 것과 (2,2) 에서 (3,2) 로 가는 것은 같은 일입니다. 상대 위치로 주면 정책이 그 사실을 따로 배울 필요가 없습니다. 로봇에서는 한 걸음 더 나가 **몸통 좌표계** 로 표현합니다(섹터 0 의 투영 중력). 공은 계속 구르므로 "몸통 좌표계" 가 의미가 없어서 세계 좌표계를 그대로 씁니다.

**왜 직전 액션을 넣는가?** 보상에 "액션을 급하게 바꾸지 말라" 는 항이 있습니다. 직전에 무엇을 냈는지 모르면 그 항을 줄일 방법이 없습니다.

### 문제 7-1. 관측 함수

모든 MDP 함수는 `env` 를 첫 인자로 받고 `(N, d)` 텐서를 돌려줍니다. 쓸 수 있는 것:

- `env.sim_adapter.root_pos_w`, `root_lin_vel_w`, `root_ang_vel_w`, `geom_friction`
- `env.command_manager.get_command(name)`
- `env.action_manager.action`

돌려주는 텐서는 시뮬레이터 메모리의 **뷰** 가 아니어야 합니다. 뷰를 그대로 돌려주면 나중에 물리가 한 스텝 가면서 값이 바뀝니다. 연산 결과(새 텐서)를 돌려주거나 `.clone()` 합니다.


In [ ]:
def ball_lin_vel(env) -> torch.Tensor:
    """(N, 2) 공의 세계 좌표계 선속도 xy."""
    return env.sim_adapter.root_lin_vel_w[:, :2].clone()


def ball_ang_vel(env) -> torch.Tensor:
    """(N, 3) 공의 세계 좌표계 각속도."""
    return env.sim_adapter.root_ang_vel_w


def goal_rel_pos(env, command_name: str = "goal") -> torch.Tensor:
    """(N, 2) 목적지 − 공 위치 (xy)."""
    return env.command_manager.get_command(command_name) - env.sim_adapter.root_pos_w[:, :2]


def last_action(env) -> torch.Tensor:
    """(N, action_dim) 직전에 낸 raw 액션."""
    return env.action_manager.action.clone()


def ball_friction(env) -> torch.Tensor:
    """(N, 1) 공의 마찰계수 (critic 전용)."""
    return env.sim_adapter.geom_friction.unsqueeze(-1).clone()


### 문제 7-2. `ObservationManager`

term 하나를 계산하는 순서는 World 의 코드 순서(`observation_manager.py` 의 `compute_group`)를 따릅니다.

```
func(env, **params)  →  noise (그룹의 enable_corruption 이 켜져 있을 때만)  →  clip  →  scale
```

World 에는 이 뒤에 EMA 필터와 이력(history) 쌓기가 더 있습니다. 여기서는 뺍니다.

- 노이즈는 `UniformNoiseCfg(n_min, n_max)`: 값에 `U(n_min, n_max)` 를 **더합니다**.
- 그룹의 `concatenate_terms` 가 켜져 있으면 term 들을 마지막 축으로 이어 붙여 `(N, 합계)` 텐서 하나로, 꺼져 있으면 `{이름: 텐서}` 사전으로 돌려줍니다.
- `compute()` 는 `{그룹 이름: 결과}` 사전을 돌려줍니다.

`__init__` 은 주어져 있습니다. 그룹마다 term 의 이름·설정·함수 목록과 차원을 준비해 둡니다.


In [ ]:
@configclass
class UniformNoiseCfg:
    n_min: float = -0.0
    n_max: float = 0.0


@configclass
class ObservationTermCfg(ManagerTermBaseCfg):
    noise: UniformNoiseCfg = None
    clip: tuple = None
    scale: float = None


@configclass
class ObservationGroupCfg:
    concatenate_terms: bool = True
    enable_corruption: bool = False


class ObservationManager:
    """(World: diden_env/managers/observation_manager.py)"""

    def __init__(self, cfg, env):
        self._env = env
        self._groups: dict[str, list[tuple[str, ObservationTermCfg, Callable]]] = {}
        self._group_cfgs: dict[str, ObservationGroupCfg] = {}
        self.group_term_dims: dict[str, dict[str, int]] = {}
        for f in fields(cfg):
            group_cfg = getattr(cfg, f.name)
            if group_cfg is None:
                continue
            terms = [(name, tc, resolve_term_func(tc, env)) for name, tc in iter_terms(group_cfg)]
            self._groups[f.name] = terms
            self._group_cfgs[f.name] = group_cfg
            self.group_term_dims[f.name] = {name: int(fn(env, **tc.params).shape[-1]) for name, tc, fn in terms}

    @property
    def group_obs_dim(self) -> dict[str, int]:
        return {g: sum(d.values()) for g, d in self.group_term_dims.items()}

    def compute_group(self, group_name: str):
        group_cfg = self._group_cfgs[group_name]
        out = {}
        for name, term_cfg, fn in self._groups[group_name]:
            obs = fn(self._env, **term_cfg.params)
            if group_cfg.enable_corruption and term_cfg.noise is not None:
                obs = obs + sample_uniform(term_cfg.noise.n_min, term_cfg.noise.n_max, obs.shape, obs.device)
            if term_cfg.clip is not None:
                obs = obs.clamp(term_cfg.clip[0], term_cfg.clip[1])
            if term_cfg.scale is not None:
                obs = obs * term_cfg.scale
            out[name] = obs
        if group_cfg.concatenate_terms:
            return torch.cat(list(out.values()), dim=-1)
        return out

    def compute(self) -> dict:
        return {g: self.compute_group(g) for g in self._groups}

    def reset(self, env_ids: torch.Tensor) -> dict:
        return {}


In [ ]:
@configclass
class ObservationsCfg:
    @configclass
    class ActorCfg(ObservationGroupCfg):
        enable_corruption: bool = True
        ball_lin_vel = ObservationTermCfg(func=ball_lin_vel, noise=UniformNoiseCfg(-0.05, 0.05))
        ball_ang_vel = ObservationTermCfg(func=ball_ang_vel, noise=UniformNoiseCfg(-0.2, 0.2))
        goal_rel_pos = ObservationTermCfg(func=goal_rel_pos, params={"command_name": "goal"}, noise=UniformNoiseCfg(-0.02, 0.02))
        last_action = ObservationTermCfg(func=last_action)

    @configclass
    class CriticCfg(ObservationGroupCfg):
        ball_lin_vel = ObservationTermCfg(func=ball_lin_vel)
        ball_ang_vel = ObservationTermCfg(func=ball_ang_vel)
        goal_rel_pos = ObservationTermCfg(func=goal_rel_pos, params={"command_name": "goal"})
        last_action = ObservationTermCfg(func=last_action)
        ball_friction = ObservationTermCfg(func=ball_friction)

    actor: ActorCfg = ActorCfg()
    critic: CriticCfg = CriticCfg()


torch.manual_seed(0)
fake = _FakeEnv(num_envs=1024)
_all = torch.arange(1024, device=DEVICE)
fake.command_manager = CommandManager(CommandsCfg(), fake)
fake.command_manager.reset(_all)
fake.action_manager = ActionManager(ActionsCfg(), fake)
fake.action_manager.process_action(torch.randn(1024, 2, device=DEVICE))
fake.sim_adapter.write_root_state(_all, torch.tensor([[0.5, -0.5, 0.2]], device=DEVICE).expand(1024, 3),
                                  torch.tensor([[1.0, 0, 0, 0]], device=DEVICE).expand(1024, 4),
                                  torch.tensor([[1.0, 2.0, 0.0]], device=DEVICE).expand(1024, 3),
                                  torch.tensor([[0.0, 0.0, 3.0]], device=DEVICE).expand(1024, 3))
fake.sim_adapter.write_geom_friction(_all, torch.full((1024,), 0.7, device=DEVICE))
fake.observation_manager = ObservationManager(ObservationsCfg(), fake)
_om = fake.observation_manager
check("그룹별 차원: actor 9, critic 10", _om.group_obs_dim == {"actor": 9, "critic": 10})
check("term 순서 = 정의 순서", list(_om.group_term_dims["actor"]) == ["ball_lin_vel", "ball_ang_vel", "goal_rel_pos", "last_action"])
_obs = _om.compute()
check("모양", _obs["actor"].shape == (1024, 9) and _obs["critic"].shape == (1024, 10))
_c = _obs["critic"]
check_close("critic: 선속도", _c[:, 0:2], torch.tensor([[1.0, 2.0]]).expand(1024, 2))
check_close("critic: 각속도", _c[:, 2:5], torch.tensor([[0.0, 0.0, 3.0]]).expand(1024, 3))
check_close("critic: 목적지 상대 위치", _c[:, 5:7], fake.command_manager.get_command("goal") - torch.tensor([[0.5, -0.5]], device=DEVICE))
check_close("critic: 직전 액션", _c[:, 7:9], fake.action_manager.action)
check_close("critic: 마찰", _c[:, 9], torch.full((1024,), 0.7))
check_close("critic 은 부를 때마다 같음 (노이즈 없음)", _om.compute()["critic"], _c, atol=0)
_diff = _obs["actor"] - _c[:, :9]
check("actor: 선속도 노이즈가 ±0.05 안이고 실제로 섞임", _diff[:, 0:2].abs().max() <= 0.05 and _diff[:, 0:2].abs().mean() > 0.01)
check("actor: 각속도 노이즈 ±0.2", _diff[:, 2:5].abs().max() <= 0.2 and _diff[:, 2:5].abs().mean() > 0.05)
check("actor: 직전 액션에는 노이즈 없음", _diff[:, 7:9].abs().max() == 0)
check("관측은 시뮬레이터 메모리의 뷰가 아님", _obs["critic"].data_ptr() != fake.sim.qvel.data_ptr())
fake.sim.qvel[:, 0] = 99.0
check("그래서 물리 상태가 바뀌어도 이미 만든 관측은 그대로", (_c[:, 0] == 1.0).all())


**실제 코드에서는** — Spider 의 actor 관측(`magnet_locomotion_mdp.py`)

| 항 | 노이즈 | 비고 |
|---|---|---|
| `imu_ang_vel` | ±0.05, 에피소드 편향 ±0.3 | 자이로. 편향은 에피소드 내내 같은 값 |
| `projected_gravity` | 기울기 ±0.05 rad, 편향 ±0.08 rad | 섹터 0 Part 7 |
| `dof_pos`, `dof_vel` | ±0.05, ±0.5 | 엔코더 |
| `relative_foot_pos` | ±0.06 | 정기구학으로 계산 (섹터 0 Part 5) |
| `gait_phi` | — | 보행 위상의 cos/sin |
| `dof_vel_history`, `dof_error_history` | | 최근 3개, 4스텝 간격 |
| `previous_joint_actions`, `previous_magnetic_action` | — | 최근 2개 |
| `command` | — | 속도 명령 |

우리 것과 다른 점: (1) 매 스텝 새로 뽑는 노이즈 외에 **에피소드 내내 고정된 편향** 이 있고, (2) 센서 종류별로 **지연** 이 있고(`delay group`), (3) **이력** 을 쌓아 넣고, (4) 몸통 선속도는 actor 에 없습니다. 선속도는 로봇에서 직접 잴 수 없어서, 별도의 추정기 네트워크가 actor 관측으로부터 추정하고 그 출력을 정책 입력에 덧붙입니다(`diden_rl/runners/on_policy_estimator_runner.py`). 우리 환경에서 `ball_lin_vel` 을 actor 에 그냥 준 것은 단순화입니다.

World 의 관측에는 `scale` 을 쓰는 항이 하나도 없습니다. 대신 네트워크 입구에서 관측 전체를 실행 평균·분산으로 정규화합니다(Part 12).


## Part 8 — `RewardManager`: 무엇을 잘한 것으로 칠 것인가

보상은 스텝마다 환경마다 숫자 하나입니다. World 는 이 숫자를 여러 항으로 나눠 적고 다음 식으로 합칩니다(`diden_env/managers/reward_manager.py`).

$$r = \Big(\sum_{i \in \text{add}} v_i\Big) \times \Big(\prod_{j \in \text{mul}} v_j\Big), \qquad v = \text{transformation}\big(w \cdot \text{raw} \cdot \Delta t\big)$$

- **add 항** 은 "잘한 일" 입니다. 양수이고 더해집니다. Spider 에서는 속도 추종, 보행 위상 추종의 세 개뿐입니다.
- **mul 항** 은 "벌점" 입니다. `transformation` 으로 `small_exp(x) = exp(0.2x)` 를 써서 (0, 1] 범위의 **배율** 로 바꾼 뒤 곱합니다. 벌점이 없으면 1, 클수록 0 에 가까워집니다. Spider 에는 이런 항이 스무 개 넘게 있습니다(발 미끄럼, 몸통 흔들림, 관절 속도·가속도, 액션 변화율, 토크 …).
- $\Delta t$ 를 곱하는 이유는 정책 주기를 바꿔도 "1초 동안 받는 보상" 이 같게 하기 위해서입니다.

### 먼저 예상해 보기

벌점을 더하지 않고 **곱하는** 이유가 무엇일까요? 다음 두 설계를 비교해 봅니다.

- (가) $r = \text{추종} - \text{벌점}$
- (나) $r = \text{추종} \times e^{-\text{벌점}}$

공이 경기장 밖으로 나가면 에피소드가 끝난다고 합시다(Part 9). 학습 초반, 정책이 아직 목적지에 가는 법을 몰라 추종 보상이 0 에 가깝고 벌점만 쌓이는 상황에서, (가) 의 정책은 무엇을 배울까요?

<details><summary>답</summary>

(가) 에서는 스텝마다 보상이 음수입니다. 에피소드가 빨리 끝날수록 음수를 덜 받으므로, 정책은 **경기장 밖으로 빨리 나가는 법** 을 배웁니다. 로봇이라면 일부러 넘어지는 것입니다. (나) 에서는 보상이 항상 0 이상이라 살아 있는 편이 언제나 이득이고, 벌점은 "추종 보상을 얼마나 깎는가" 로만 작용합니다. 추종을 전혀 못 하면 벌점도 의미가 없어지므로, 정책은 먼저 추종을 배우고 그다음에 매끄럽게 하는 법을 배웁니다.
</details>

우리 보상:

| 항 | 종류 | weight | raw |
|---|---|---|---|
| `goal_tracking_coarse` | add | 1.0 | $\exp(-d^2 / 2.0^2)$ — 멀리서도 방향을 알려 준다 |
| `goal_tracking_fine` | add | 2.0 | $\exp(-d^2 / 0.3^2)$ — 정확히 도착해 머물러야 받는다 |
| `action_rate` | mul | −5.0 | $\lVert a_t - a_{t-1} \rVert^2$ |
| `action_magnitude` | mul | −1.0 | $\lVert a_t \rVert^2$ |

$d$ 는 공과 목적지 사이의 거리입니다. `tracking_exp(error_sq, sigma) = exp(−error_sq / sigma²)` 는 World 의 추종 보상들이 공유하는 커널입니다(`diden_common/reward_transforms.py`).

### 문제 8-1. 보상 함수와 변환


In [ ]:
def small_exp(y: torch.Tensor) -> torch.Tensor:
    """exp(0.2·y): (음수인) 벌점을 (0, 1] 의 배율로. (World: reward_transforms.small_exp)"""
    return torch.exp(0.2 * y)


def tracking_exp(error_sq: torch.Tensor, sigma: float) -> torch.Tensor:
    """exp(−error_sq / sigma²): 오차 제곱을 (0, 1] 의 보상으로. 1 = 정확히 일치."""
    return torch.exp(-error_sq / sigma**2)


def goal_distance(env, command_name: str = "goal") -> torch.Tensor:
    """(N,) 공과 목적지 사이의 수평 거리. (보상, 종료, 평가에서 같이 쓴다)"""
    return (env.command_manager.get_command(command_name) - env.sim_adapter.root_pos_w[:, :2]).norm(dim=-1)


def goal_tracking(env, sigma: float, command_name: str = "goal") -> torch.Tensor:
    """(N,) tracking_exp(d², sigma)."""
    return tracking_exp(goal_distance(env, command_name) ** 2, sigma)


def action_rate_l2(env) -> torch.Tensor:
    """(N,) ‖a_t − a_{t−1}‖²."""
    return (env.action_manager.action - env.action_manager.prev_action).square().sum(dim=-1)


def action_l2(env) -> torch.Tensor:
    """(N,) ‖a_t‖²."""
    return env.action_manager.action.square().sum(dim=-1)


### 문제 8-2. `RewardManager`

`compute(dt)` 는 `(N,)` 보상을 돌려줍니다.

- 항마다 `value = transformation(weight * raw * dt)`
- `operation == "add"` 면 합에 더하고, `"mul"` 이면 곱에 곱한다
- 최종 `reward = 합 × 곱`
- 기록용으로 `self._episode_sums[name] += value`, `self._step_values[name] = value` 를 남긴다

`reset(env_ids)` 는 리셋되는 환경들의 에피소드 합계를 평균 내어 `{"Episode_Reward/<이름>": 값}` 사전으로 돌려주고 합계를 0 으로 되돌립니다. 이 사전이 학습 로그가 됩니다. 보상 항이 많아지면 "지금 정책이 어떤 항에서 점수를 잃고 있는가" 를 이 로그로 봅니다.


In [ ]:
@configclass
class RewardTermCfg(ManagerTermBaseCfg):
    weight: float = 0.0
    operation: str = "add"                        # "add" | "mul"
    transformation: Callable = None               # None 이면 그대로


class RewardManager:
    """(World: diden_env/managers/reward_manager.py)"""

    def __init__(self, cfg, env):
        self._env = env
        self._terms = [(name, tc, resolve_term_func(tc, env)) for name, tc in iter_terms(cfg)]
        self._episode_sums = {name: torch.zeros(env.num_envs, device=env.device) for name, _, _ in self._terms}
        self._step_values = {}
        self._reward_buf = torch.zeros(env.num_envs, device=env.device)

    def compute(self, dt: float) -> torch.Tensor:
        add = torch.zeros(self._env.num_envs, device=self._env.device)
        mul = torch.ones(self._env.num_envs, device=self._env.device)
        for name, term_cfg, fn in self._terms:
            raw = fn(self._env, **term_cfg.params)
            value = term_cfg.weight * raw * dt
            if term_cfg.transformation is not None:
                value = term_cfg.transformation(value)
            if term_cfg.operation == "add":
                add = add + value
            elif term_cfg.operation == "mul":
                mul = mul * value
            else:
                raise ValueError(term_cfg.operation)
            self._episode_sums[name] += value
            self._step_values[name] = value
        self._reward_buf = add * mul
        return self._reward_buf

    def reset(self, env_ids: torch.Tensor) -> dict:
        log = {}
        for name in self._episode_sums:
            log[f"Episode_Reward/{name}"] = self._episode_sums[name][env_ids].mean().item()
            self._episode_sums[name][env_ids] = 0.0
        return log


In [ ]:
@configclass
class RewardsCfg:
    goal_tracking_coarse = RewardTermCfg(func=goal_tracking, weight=1.0, params={"sigma": 2.0, "command_name": "goal"})
    goal_tracking_fine = RewardTermCfg(func=goal_tracking, weight=2.0, params={"sigma": 0.3, "command_name": "goal"})
    action_rate = RewardTermCfg(func=action_rate_l2, weight=-5.0, operation="mul", transformation=small_exp)
    action_magnitude = RewardTermCfg(func=action_l2, weight=-1.0, operation="mul", transformation=small_exp)

check_close("small_exp", small_exp(torch.tensor([0.0, -5.0])), [1.0, math.exp(-1.0)])
check_close("tracking_exp", tracking_exp(torch.tensor([0.0, 4.0]), 2.0), [1.0, math.exp(-1.0)])

fake = _FakeEnv(num_envs=4)
_all = torch.arange(4, device=DEVICE)
fake.command_manager = CommandManager(CommandsCfg(), fake)
fake.command_manager.reset(_all)
fake.command_manager.get_term("goal").goal_pos_w[:] = torch.tensor([[1.0, 0.0]], device=DEVICE)     # 목적지를 (1, 0) 에 고정
fake.action_manager = ActionManager(ActionsCfg(), fake)
# 환경 0: 목적지 위·액션 0 / 1: 2 m 떨어짐 / 2: 목적지 위·액션 (1,0) 유지 / 3: 목적지 위·액션을 (0,0)→(1,1) 로 급변
_pos = torch.tensor([[1.0, 0.0, 0.2], [1.0, 2.0, 0.2], [1.0, 0.0, 0.2], [1.0, 0.0, 0.2]], device=DEVICE)
_zero3 = torch.zeros(4, 3, device=DEVICE)
fake.sim_adapter.write_root_state(_all, _pos, torch.tensor([[1.0, 0, 0, 0]], device=DEVICE).expand(4, 4), _zero3, _zero3)
fake.action_manager.process_action(torch.tensor([[0.0, 0.0], [0.0, 0.0], [1.0, 0.0], [0.0, 0.0]], device=DEVICE))
fake.action_manager.process_action(torch.tensor([[0.0, 0.0], [0.0, 0.0], [1.0, 0.0], [1.0, 1.0]], device=DEVICE))
fake.reward_manager = RewardManager(RewardsCfg(), fake)
_dt = 0.02
_r = fake.reward_manager.compute(_dt)
check_close("거리", goal_distance(fake), [0.0, 2.0, 0.0, 0.0])
check_close("환경 0: 목적지 위, 벌점 없음 → (1 + 2)·dt", _r[0], 3.0 * _dt)
check_close("환경 1: 2 m 밖 → (e^-1 + 2·e^(-4/0.09))·dt", _r[1], (math.exp(-1.0) + 2.0 * math.exp(-4.0 / 0.09)) * _dt, atol=1e-7)
check_close("환경 2: 액션 크기 벌점 → 3·dt·exp(0.2·(−1)·1·dt)", _r[2], 3.0 * _dt * math.exp(0.2 * -1.0 * 1.0 * _dt), atol=1e-7)
check_close("환경 3: 변화율·크기 벌점 둘 다 (‖Δa‖²=2, ‖a‖²=2)", _r[3],
            3.0 * _dt * math.exp(0.2 * -5.0 * 2.0 * _dt) * math.exp(0.2 * -1.0 * 2.0 * _dt), atol=1e-7)
check("보상은 항상 0 이상", (_r >= 0).all())
fake.reward_manager.compute(_dt)
_log = fake.reward_manager.reset(torch.tensor([0], device=DEVICE))
check_close("에피소드 합 기록 (환경 0, 2 스텝)", _log["Episode_Reward/goal_tracking_fine"], 2 * 2.0 * _dt, atol=1e-7)
check("리셋된 환경만 합계가 0 으로", fake.reward_manager._episode_sums["goal_tracking_fine"][0] == 0
      and fake.reward_manager._episode_sums["goal_tracking_fine"][2] > 0)


**실제 코드에서는**

- 가중치가 훨씬 큽니다(`tracking_lin_vel` 100, `foot_slip` 800, `base_height` 6000 …). 여기에 `dt = 0.01` 과 `small_exp` 의 0.2 가 곱해지므로 숫자만 보고 크기를 비교하면 안 됩니다. "raw 가 얼마일 때 배율이 얼마가 되는가" 로 읽습니다.
- **regime**: 항마다 "어떤 상황에서만 켜지는가" 를 지정할 수 있습니다(예: 정상 접촉 중일 때만). 꺼진 환경에는 add 면 0, mul 이면 1 을 넣습니다.
- **스케줄**: 관절 액션 평활도 벌점은 가중치가 학습 5000~10000 iteration 에 걸쳐 0.8 → 10 으로 커지고, `joint_torque` 벌점도 0.2 → 2.5 로 커집니다. 처음에는 걷는 것부터 배우고 나중에 매끄러움과 효율을 다듬게 하는 커리큘럼입니다.
- 보상 설계를 바꿀 때 가장 먼저 보는 것이 `Episode_Reward/*` 로그입니다. 어떤 mul 항의 배율이 0.1 근처에 붙어 있으면 그 항이 다른 모든 학습 신호를 지우고 있다는 뜻입니다.


## Part 9 — `TerminationManager`: 에피소드는 언제 끝나는가

에피소드가 끝나는 이유는 두 종류이고, **학습에서 다르게 취급합니다.**

| | 뜻 | 예 | 가치 추정에서 |
|---|---|---|---|
| **terminated** (종료) | 과제가 실패했다. 이 뒤는 없다. | 로봇이 넘어짐, 공이 경기장 밖으로 나감 | 이후 보상 = 0 |
| **time_out** (시간 초과, truncated) | 잘하고 있었는데 시간이 다 됐다. | 에피소드 길이 도달 | 계속했다면 받았을 보상을 추정해 더해 준다 (bootstrap) |

둘을 구분하지 않으면 정책은 "8초가 되면 갑자기 보상이 끊긴다" 를 실패와 똑같이 받아들입니다. 그러면 관측에 시간이 없는데도 가치가 시간에 따라 달라져서 가치 함수 학습이 흔들립니다. 이 구분을 실제로 쓰는 곳은 Part 14 의 PPO 입니다.

World 의 구현(`termination_manager.py`)은 항마다 불리언 `(N,)` 을 받아 `time_out=True` 인 항은 truncated 쪽에, 나머지는 terminated 쪽에 OR 로 모읍니다. Spider 의 종료 항은 `time_out`, NaN 발생, 몸통 속도 한계 초과, 금지된 부위의 접촉, 기울기 60° 초과입니다.

우리 종료 항:

- `time_out`: `env.episode_length_buf >= env.max_episode_length`
- `out_of_bounds`: 공의 |x| 또는 |y| 가 `limit`(5 m)을 넘음

### 문제 9-1


In [ ]:
def time_out(env) -> torch.Tensor:
    """(N,) bool. 에피소드 길이에 도달했는가."""
    return env.episode_length_buf >= env.max_episode_length


def out_of_bounds(env, limit: float) -> torch.Tensor:
    """(N,) bool. 공이 한 변 2·limit 인 정사각형 밖으로 나갔는가."""
    return (env.sim_adapter.root_pos_w[:, :2].abs() > limit).any(dim=-1)


@configclass
class TerminationTermCfg(ManagerTermBaseCfg):
    time_out: bool = False


class TerminationManager:
    """(World: diden_env/managers/termination_manager.py)"""

    def __init__(self, cfg, env):
        self._env = env
        self._terms = [(name, tc, resolve_term_func(tc, env)) for name, tc in iter_terms(cfg)]
        n, dev = env.num_envs, env.device
        self._terminated_buf = torch.zeros(n, dtype=torch.bool, device=dev)
        self._truncated_buf = torch.zeros(n, dtype=torch.bool, device=dev)
        self._term_dones = {name: torch.zeros(n, dtype=torch.bool, device=dev) for name, _, _ in self._terms}

    @property
    def terminated(self) -> torch.Tensor:
        return self._terminated_buf

    @property
    def time_outs(self) -> torch.Tensor:
        return self._truncated_buf

    def compute(self) -> torch.Tensor:
        """(N,) bool: 이번 스텝에 리셋해야 하는 환경."""
        self._terminated_buf[:] = False
        self._truncated_buf[:] = False
        for name, term_cfg, fn in self._terms:
            value = fn(self._env, **term_cfg.params)
            if term_cfg.time_out:
                self._truncated_buf |= value
            else:
                self._terminated_buf |= value
            self._term_dones[name][:] = value
        return self._terminated_buf | self._truncated_buf

    def reset(self, env_ids: torch.Tensor) -> dict:
        """리셋되는 환경들이 어떤 이유로 끝났는지의 비율을 기록한다."""
        log = {}
        for name, done in self._term_dones.items():
            log[f"Episode_Termination/{name}"] = done[env_ids].float().mean().item()
            done[env_ids] = False
        return log


In [ ]:
@configclass
class TerminationsCfg:
    time_out = TerminationTermCfg(func=time_out, time_out=True)
    out_of_bounds = TerminationTermCfg(func=out_of_bounds, params={"limit": 5.0})

fake = _FakeEnv(num_envs=4)
_all = torch.arange(4, device=DEVICE)
fake.termination_manager = TerminationManager(TerminationsCfg(), fake)
_pos = torch.tensor([[0.0, 0.0, 0.2], [5.1, 0.0, 0.2], [0.0, -5.1, 0.2], [5.1, 0.0, 0.2]], device=DEVICE)
_zero3 = torch.zeros(4, 3, device=DEVICE)
fake.sim_adapter.write_root_state(_all, _pos, torch.tensor([[1.0, 0, 0, 0]], device=DEVICE).expand(4, 4), _zero3, _zero3)
fake.episode_length_buf[:] = torch.tensor([399, 10, 10, 400], device=DEVICE)
_done = fake.termination_manager.compute()
check("리셋 대상", _done.tolist() == [False, True, True, True])
check("terminated: 밖으로 나간 환경", fake.termination_manager.terminated.tolist() == [False, True, True, True])
check("time_outs: 길이에 도달한 환경", fake.termination_manager.time_outs.tolist() == [False, False, False, True])
fake.episode_length_buf[0] = 400
fake.termination_manager.compute()
check("399 → 400 에서 time_out", fake.termination_manager.time_outs.tolist() == [True, False, False, True])
_log = fake.termination_manager.reset(_all)
check_close("종료 이유 비율 기록", [_log["Episode_Termination/time_out"], _log["Episode_Termination/out_of_bounds"]], [0.5, 0.75])


환경 3 은 두 조건이 **동시에** 참입니다. 이런 경우 World 는 terminated 를 우선합니다. 시간 초과로 보고 bootstrap 하는 대상은 `time_out & ~terminated` 인 환경뿐입니다(`manager_based_rl_env.py` 의 `timeout_env_ids`). Part 11 에서 같은 식을 씁니다.


## Part 10 — `EventManager`: 리셋과 도메인 랜덤화

**이벤트** 는 "정책과 무관하게 환경에 일어나는 일" 입니다. 언제 일어나는가를 **mode** 로 나눕니다. World 의 mode(`diden_env/managers/event_manager.py`)는 다음과 같고, 필요하면 자기 mode 를 만들어 환경에서 직접 불러도 된다고 되어 있습니다.

| mode | 언제 | Spider 에서의 예 |
|---|---|---|
| `startup` | 학습 시작 때 한 번 | (Spider 는 쓰지 않음) |
| `reset` | 환경이 리셋될 때, 그 환경에만 | 초기 자세, PD 게인 ±10 %, 마찰 0.25~0.4, 링크 질량 ×0.8~1.5, 무게중심 이동, 로터 관성 |
| `physics_step` | 물리 서브스텝마다 | 자석 상태기계, 관절 stick-slip 마찰 |

**도메인 랜덤화(DR)** 는 시뮬레이터의 물리 파라미터를 환경마다, 에피소드마다 다르게 뽑는 것입니다. 실제 로봇의 마찰, 질량, 모터 특성을 정확히 알 수 없으므로 "이 범위 어딘가에 있을 것" 이라고 보고 그 범위 전체에서 동작하는 정책을 학습시킵니다. 범위가 좁으면 실제 로봇에서 실패하고, 넓으면 학습이 어려워지고 정책이 조심스러워집니다.

이벤트 함수의 서명은 `func(env, env_ids, **params)` 입니다. 관측·보상 함수와 달리 `env_ids` 를 받고 아무것도 돌려주지 않습니다.

우리 이벤트:

| 항 | mode | 하는 일 |
|---|---|---|
| `randomize_friction` | `reset` | 공의 마찰계수를 0.3~1.0 에서 새로 뽑는다 |
| `reset_ball` | `reset` | 공을 ±2 m 안 임의 위치에, ±0.5 m/s 의 임의 속도로 놓는다 |
| `push_ball` | `interval` (우리가 만드는 mode) | 3~5 초마다 공을 ±1 m/s 로 걷어찬다 |

`interval` 은 World 의 `EventManager` 에는 없는 mode 입니다. World 에는 `push_robot` 함수가 있지만(`diden_env/mdp/events/reset.py`) Spider 설정에 연결되어 있지 않습니다. 여기서는 "외란을 견디는 정책" 을 보기 위해 넣습니다.

### 문제 10-1. 이벤트 함수

`reset_ball_uniform` 에서 공을 굴러가는 중인 상태로 놓으려면 선속도 $v$ 에 맞는 각속도도 줘야 합니다. 미끄러지지 않고 구르는 공은 $\omega = \hat z \times v / r$, 즉 $(\omega_x, \omega_y) = (-v_y, v_x) / r$ 입니다. 각속도를 0 으로 두면 리셋 직후 공이 바닥에서 미끄러지다가 구르기 시작해, 정책이 보는 첫 몇 스텝이 이상해집니다.


In [ ]:
def randomize_ball_friction(env, env_ids: torch.Tensor, friction_range: tuple) -> None:
    friction = sample_uniform(*friction_range, (len(env_ids),), env.device)
    env.sim_adapter.write_geom_friction(env_ids, friction)


def reset_ball_uniform(env, env_ids: torch.Tensor, pos_range: tuple, lin_vel_range: tuple) -> None:
    """공을 임의 위치·속도로 놓는다. 높이는 반지름, 자세는 단위 쿼터니언, 각속도는 구름 조건에 맞춘다."""
    n, dev, r = len(env_ids), env.device, env.sim_adapter.radius
    pos = torch.zeros(n, 3, device=dev)
    pos[:, :2] = sample_uniform(*pos_range, (n, 2), dev)
    pos[:, 2] = r
    quat = torch.zeros(n, 4, device=dev)
    quat[:, 0] = 1.0
    lin_vel = torch.zeros(n, 3, device=dev)
    lin_vel[:, :2] = sample_uniform(*lin_vel_range, (n, 2), dev)
    ang_vel = torch.stack([-lin_vel[:, 1], lin_vel[:, 0], torch.zeros(n, device=dev)], dim=-1) / r
    env.sim_adapter.write_root_state(env_ids, pos, quat, lin_vel, ang_vel)


def push_ball(env, env_ids: torch.Tensor, velocity_range: tuple) -> None:
    """공의 선속도 xy 에 임의의 값을 더한다 (걷어차기)."""
    ad = env.sim_adapter
    lin_vel = ad.root_lin_vel_w[env_ids].clone()
    lin_vel[:, :2] += sample_uniform(*velocity_range, (len(env_ids), 2), env.device)
    ad.write_root_state(env_ids, ad.root_pos_w[env_ids].clone(), ad.root_quat_w[env_ids].clone(), lin_vel, ad.root_ang_vel_w[env_ids])


### 문제 10-2. `EventManager`

- `apply(mode, env_ids=None, dt=None)`
  - `startup`, `reset` 등: 그 mode 의 항을 순서대로 `func(env, env_ids, **params)`. `env_ids` 가 `None` 이면 전체 환경.
  - `interval`: 항마다 환경별 남은 시간 `self._interval_time_left[name]` 이 있다. `dt` 만큼 줄이고, 0 이하가 된 환경에만 함수를 부른 뒤, 그 환경의 남은 시간을 `interval_range_s` 에서 다시 뽑는다.
- `reset(env_ids)`: 리셋된 환경의 interval 타이머를 다시 뽑는다.


In [ ]:
@configclass
class EventTermCfg(ManagerTermBaseCfg):
    mode: str = "reset"
    interval_range_s: tuple = None            # mode="interval" 일 때만


class EventManager:
    """(World: diden_env/managers/event_manager.py)"""

    def __init__(self, cfg, env):
        self._env = env
        self._mode_terms: dict[str, list[tuple[str, EventTermCfg, Callable]]] = {}
        self._interval_time_left: dict[str, torch.Tensor] = {}
        for name, tc in iter_terms(cfg):
            self._mode_terms.setdefault(tc.mode, []).append((name, tc, resolve_term_func(tc, env)))
            if tc.mode == "interval":
                self._interval_time_left[name] = sample_uniform(*tc.interval_range_s, (env.num_envs,), env.device)

    @property
    def available_modes(self) -> list[str]:
        return list(self._mode_terms)

    def apply(self, mode: str, env_ids: torch.Tensor = None, dt: float = None) -> None:
        if mode not in self._mode_terms:
            return
        for name, term_cfg, fn in self._mode_terms[mode]:
            if mode == "interval":
                time_left = self._interval_time_left[name]
                time_left -= dt
                ids = (time_left <= 0.0).nonzero(as_tuple=False).squeeze(-1)
                if len(ids) > 0:
                    fn(self._env, ids, **term_cfg.params)
                    time_left[ids] = sample_uniform(*term_cfg.interval_range_s, (len(ids),), self._env.device)
            else:
                ids = env_ids if env_ids is not None else torch.arange(self._env.num_envs, device=self._env.device)
                fn(self._env, ids, **term_cfg.params)

    def reset(self, env_ids: torch.Tensor) -> dict:
        for name, term_cfg, _ in self._mode_terms.get("interval", []):
            self._interval_time_left[name][env_ids] = sample_uniform(*term_cfg.interval_range_s, (len(env_ids),), self._env.device)
        return {}


In [ ]:
@configclass
class EventsCfg:
    randomize_friction = EventTermCfg(func=randomize_ball_friction, mode="reset", params={"friction_range": (0.3, 1.0)})
    reset_ball = EventTermCfg(func=reset_ball_uniform, mode="reset", params={"pos_range": (-2.0, 2.0), "lin_vel_range": (-0.5, 0.5)})
    push_ball = EventTermCfg(func=push_ball, mode="interval", interval_range_s=(3.0, 5.0), params={"velocity_range": (-1.0, 1.0)})

torch.manual_seed(0)
fake = _FakeEnv(num_envs=4096)
_all = torch.arange(4096, device=DEVICE)
fake.event_manager = EventManager(EventsCfg(), fake)
check("mode 목록", set(fake.event_manager.available_modes) == {"reset", "interval"})
_ad = fake.sim_adapter
_some = torch.arange(0, 4096, 2, device=DEVICE)                 # 짝수 환경만 리셋
fake.event_manager.apply("reset", env_ids=_some)
fake.sim.forward()
check("리셋된 환경: 위치 ±2, 높이 = 반지름", _ad.root_pos_w[_some, :2].abs().max() <= 2.0 and (_ad.root_pos_w[_some, 2] == BALL_RADIUS).all())
check("리셋 안 된 환경은 원점 그대로", _ad.root_pos_w[1::2, :2].abs().max() == 0)
check("마찰이 0.3~1.0 로 랜덤화 (리셋된 환경만)", _ad.geom_friction[_some].min() >= 0.3 and _ad.geom_friction[_some].max() < 1.0
      and _ad.geom_friction[_some].std() > 0.15 and (_ad.geom_friction[1::2] == 1.0).all())
_v, _w = _ad.root_lin_vel_w[_some], _ad.root_ang_vel_w[_some]
check_close("구름 조건: 접촉점 속도 v + ω × (−r ẑ) = 0", _v[:, :2] - torch.stack([_w[:, 1], -_w[:, 0]], dim=-1) * BALL_RADIUS, torch.zeros(2048, 2), atol=1e-5)
_pos0 = _ad.root_pos_w[_some].clone()
for _ in range(20):
    fake.sim.step()
_slip = (_ad.root_lin_vel_w[_some, 0] - _ad.root_ang_vel_w[_some, 1] * BALL_RADIUS).abs().max().item()
check(f"그래서 리셋 직후에도 미끄러지지 않고 구른다 (접촉점 미끄럼 {_slip:.4f} m/s)", _slip < 0.02)

# interval: 3~5 초마다 걷어차기
fake.event_manager.apply("reset", env_ids=_all)
_v0 = _ad.root_lin_vel_w.clone()
for _ in range(149):                                             # 2.98 초
    fake.event_manager.apply("interval", dt=0.02)
check("3초 전에는 아무도 차이지 않음", (_ad.root_lin_vel_w == _v0).all())
for _ in range(60):                                              # 4.18 초 시점
    fake.event_manager.apply("interval", dt=0.02)
_pushed = (_ad.root_lin_vel_w != _v0).any(dim=-1).float().mean().item()
print(f"   4.18초 시점에 차인 환경 비율: {_pushed:.3f}  (기대값 (4.18−3)/2 = 0.59)")
check("일부 환경만 차임", 0.5 < _pushed < 0.7)


**실제 코드에서는**

- 리셋 자세가 훨씬 복잡합니다. Spider 는 발 위치 ±0.15 m, 몸통 ±0.05 m, roll/pitch ±15°, yaw 전 범위를 뽑은 뒤 **역기구학** 으로 그 자세에 맞는 관절각을 풉니다(`diden_env/mdp/events/reset.py`). 벽에 붙은 채로 시작해야 하므로 아무 자세나 줄 수 없습니다.
- **설정의 값이 나중에 덮어써지는 경우** 가 있습니다. Spider 설정 파일의 PD 게인 랜덤화 범위는 25~35 로 적혀 있지만 환경 설정에서 100/20 의 ±10 % 로 다시 써집니다. 액션 스케일도 0.4 로 적혀 있고 0.2 로 덮어써집니다. 설정을 읽을 때는 "마지막에 누가 쓰는가" 까지 따라가야 합니다.
- 모델 값을 환경마다 다르게 주려면 그 필드를 미리 늘려 둬야 합니다(Part 3). World 는 랜덤화할 필드 목록을 환경 생성 때 한꺼번에 늘립니다. 목록에 없는 필드를 랜덤화하려 하면 모든 환경이 같은 값을 공유하게 됩니다.


## Part 11 — `ManagerBasedRlEnv`: 전부 묶기

이제 매니저 여섯 개를 한 환경으로 묶습니다. 이 Part 의 핵심은 `step()` 안의 **순서** 입니다. World 의 `ManagerBasedRlEnv.step` (`diden_env/env/manager_based_rl_env.py`)에서 계측·로그 코드를 걷어내면 다음이 남습니다.

```
step(action):
  1. action_manager.process_action(action)
  2. 물리: decimation 번 반복 { action_manager.apply_action();  sim.step() }
  3. episode_length_buf += 1
  4. reset_buf = termination_manager.compute()          ← 종료 판정
  5. reward_buf = reward_manager.compute(dt=step_dt)    ← 보상
  6. 리셋할 환경이 있으면:
       a. 시간 초과(time_out & ~terminated) 환경의 "리셋 직전 critic 관측" 을 extras["terminal_obs"] 에 저장
       b. _reset_idx(env_ids);  sim.forward()
  7. command_manager.compute(dt=step_dt)
  8. (우리가 추가) event_manager.apply("interval", dt=step_dt)
  9. obs_buf = observation_manager.compute()            ← 관측
  return obs_buf, reward_buf, terminated, time_outs, extras
```

### 먼저 예상해 보기

1. 보상(5)을 리셋(6)보다 **먼저** 계산해야 하는 이유는?
2. 관측(9)을 리셋(6)보다 **나중에** 계산하면, 방금 끝난 환경에 대해 돌려주는 `obs` 는 어느 에피소드의 것일까요? 그래도 되는 이유는?
3. 6a 에서 리셋 직전 관측을 따로 챙기는 이유는? (Part 9 의 time_out 설명)

<details><summary>답</summary>

1. 보상은 "방금 한 행동의 결과" 에 대한 것입니다. 리셋하고 나면 공이 새 위치에 있어서 엉뚱한 상태의 보상이 계산됩니다.
2. **새 에피소드의 첫 관측** 입니다. 환경이 스스로 리셋하므로(auto-reset) 학습 루프는 멈추지 않고 그 관측으로 다음 액션을 뽑으면 됩니다. 대신 끝난 에피소드의 마지막 상태는 `obs` 에 없습니다.
3. 시간 초과는 실패가 아니므로 "계속했다면 받았을 가치" $V(s_{T})$ 를 더해 줘야 하는데, 그 $s_T$ 가 2번 때문에 `obs` 에서 사라졌습니다. 그래서 리셋하기 전에 따로 저장합니다.
</details>

`_reset_idx(env_ids)` 의 순서도 정해져 있습니다(`manager_based_rl_env.py` 의 `_reset_idx`).

```
_reset_idx(env_ids):
  1. event_manager.apply("reset", env_ids)     ← 물리 상태를 새로 쓴다
  2. 각 매니저의 reset(env_ids)                 ← 내부 버퍼를 비우고 로그를 돌려준다
       observation → action → reward → command → event → termination
  3. episode_length_buf[env_ids] = 0
```

### 설정 한 곳에 모으기

지금까지 채점 셀에서 따로 만들었던 설정 클래스를 하나로 묶습니다. World 에서 이 역할을 하는 것이 `ManagerBasedRLEnvCfg` 를 상속한 환경 설정(예: `magnet_locomotion_env_cfg.py`)이고, sim profile 이 `dt` 와 `decimation` 을 정합니다.


In [ ]:
@configclass
class SimCfg:
    dt: float = 0.005                 # 물리 스텝 [s]   (Spider: 0.001)
    device: str = DEVICE


@configclass
class SceneCfg:
    num_envs: int = 4096              # (Spider: 4096)


@configclass
class BallEnvCfg:
    seed: int = 0
    decimation: int = 4               # 정책 한 스텝 = 물리 4 스텝 → 50 Hz   (Spider: 10 → 100 Hz)
    episode_length_s: float = 8.0     # (Spider: 40)
    sim: SimCfg = SimCfg()
    scene: SceneCfg = SceneCfg()
    commands: CommandsCfg = CommandsCfg()
    actions: ActionsCfg = ActionsCfg()
    observations: ObservationsCfg = ObservationsCfg()
    rewards: RewardsCfg = RewardsCfg()
    terminations: TerminationsCfg = TerminationsCfg()
    events: EventsCfg = EventsCfg()


### 문제 11-1. 환경

`__init__` 과 속성은 주어져 있습니다. `step`, `_reset_idx`, `reset` 을 채웁니다.

- `extras` 는 사전입니다. `_reset_idx` 는 매니저들의 `reset()` 이 돌려준 로그 사전을 모아 `self.extras["log"]` 에 넣습니다.
- `extras["terminal_obs"]` 는 `{"env_ids": 텐서, "obs": critic 관측}` 이고, 시간 초과 환경이 없는 스텝에서는 키 자체가 없어야 합니다.
- `reset()` 은 모든 환경을 리셋하고 `(obs, extras)` 를 돌려줍니다.


In [ ]:
class ManagerBasedRlEnv:
    """(World: diden_env/env/manager_based_env.py + manager_based_rl_env.py + diden_mjlab/env/mjwarp_env.py)"""

    def __init__(self, cfg: BallEnvCfg):
        self.cfg = cfg
        torch.manual_seed(cfg.seed)
        # 씬 → 시뮬레이션 → 어댑터   (World: create_scene, create_sim_context, _create_sim_adapter)
        self.sim = BatchedSim(build_ball_scene(timestep=cfg.sim.dt).compile(), cfg.scene.num_envs, cfg.sim.device)
        self.sim_adapter = SimAdapter(self.sim)
        # 버퍼
        self.episode_length_buf = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.common_step_counter = 0
        self.extras: dict = {}
        self.reset_buf = torch.zeros(self.num_envs, dtype=torch.bool, device=self.device)
        self.reward_buf = torch.zeros(self.num_envs, device=self.device)
        # 매니저   (World: load_managers — event 가 가장 먼저, 그다음 command → action → observation → termination → reward)
        self.event_manager = EventManager(cfg.events, self)
        self.command_manager = CommandManager(cfg.commands, self)
        self.action_manager = ActionManager(cfg.actions, self)
        self.observation_manager = ObservationManager(cfg.observations, self)
        self.termination_manager = TerminationManager(cfg.terminations, self)
        self.reward_manager = RewardManager(cfg.rewards, self)
        self.event_manager.apply("startup")
        self.obs_buf: dict = {}

    # ── 속성 ───────────────────────────────────────────────
    @property
    def num_envs(self) -> int:
        return self.cfg.scene.num_envs

    @property
    def device(self) -> str:
        return self.cfg.sim.device

    @property
    def physics_dt(self) -> float:
        return self.cfg.sim.dt

    @property
    def step_dt(self) -> float:
        return self.cfg.sim.dt * self.cfg.decimation

    @property
    def max_episode_length_s(self) -> float:
        return self.cfg.episode_length_s

    @property
    def max_episode_length(self) -> int:
        return math.ceil(self.max_episode_length_s / self.step_dt)

    @property
    def num_actions(self) -> int:
        return self.action_manager.total_action_dim

    # ── MDP ────────────────────────────────────────────────
    def _reset_idx(self, env_ids: torch.Tensor) -> None:
        self.event_manager.apply("reset", env_ids=env_ids)
        log = {}
        for manager in (self.observation_manager, self.action_manager, self.reward_manager,
                        self.command_manager, self.event_manager, self.termination_manager):
            log.update(manager.reset(env_ids))
        self.extras["log"] = log
        self.episode_length_buf[env_ids] = 0

    def reset(self) -> tuple[dict, dict]:
        self._reset_idx(torch.arange(self.num_envs, device=self.device))
        self.sim.forward()
        self.obs_buf = self.observation_manager.compute()
        return self.obs_buf, self.extras

    def step(self, action: torch.Tensor):
        # 1. 액션 처리
        self.action_manager.process_action(action)
        # 2. 물리
        for _ in range(self.cfg.decimation):
            self.action_manager.apply_action()
            self.sim.step()
        # 3. 카운터
        self.episode_length_buf += 1
        self.common_step_counter += 1
        # 4. 종료 판정
        self.reset_buf = self.termination_manager.compute()
        terminated = self.termination_manager.terminated.clone()
        time_outs = self.termination_manager.time_outs.clone()
        # 5. 보상
        self.reward_buf = self.reward_manager.compute(dt=self.step_dt)
        # 6. 리셋
        self.extras.pop("terminal_obs", None)
        reset_env_ids = self.reset_buf.nonzero(as_tuple=False).squeeze(-1)
        if len(reset_env_ids) > 0:
            timeout_env_ids = (time_outs & ~terminated).nonzero(as_tuple=False).squeeze(-1)
            if len(timeout_env_ids) > 0:
                critic_obs = self.observation_manager.compute_group("critic")
                self.extras["terminal_obs"] = {"env_ids": timeout_env_ids, "obs": critic_obs[timeout_env_ids].clone()}
            self._reset_idx(reset_env_ids)
            self.sim.forward()
        # 7. 명령
        self.command_manager.compute(dt=self.step_dt)
        # 8. interval 이벤트
        self.event_manager.apply("interval", dt=self.step_dt)
        # 9. 관측
        self.obs_buf = self.observation_manager.compute()
        return self.obs_buf, self.reward_buf, terminated, time_outs, self.extras


In [ ]:
env = ManagerBasedRlEnv(BallEnvCfg(scene=SceneCfg(num_envs=1024)))
check("시간: 물리 0.005 s × decimation 4 = 0.02 s, 에피소드 400 스텝", abs(env.step_dt - 0.02) < 1e-12 and env.max_episode_length == 400)
obs, extras = env.reset()
check("reset 이 돌려주는 관측", obs["actor"].shape == (1024, 9) and obs["critic"].shape == (1024, 10))
check("리셋 후: 공은 ±2 안, 마찰은 0.3~1.0", env.sim_adapter.root_pos_w[:, :2].abs().max() <= 2.0 and 0.3 <= env.sim_adapter.geom_friction.min())
check("extras['log'] 에 매니저 로그", "Episode_Reward/goal_tracking_fine" in extras["log"] and "Episode_Termination/time_out" in extras["log"])

_zero = torch.zeros(1024, 2, device=DEVICE)
obs, rew, terminated, time_outs, extras = env.step(_zero)
check("step 의 반환 모양", obs["actor"].shape == (1024, 9) and rew.shape == (1024,) and terminated.dtype == torch.bool and time_outs.shape == (1024,))
check("episode_length_buf 가 1 증가", (env.episode_length_buf == 1).all())

# 액션 (1, 0) 을 10 스텝 → +x 로 가속
_v0 = env.sim_adapter.root_lin_vel_w[:, 0].clone()
for _ in range(10):
    env.step(torch.tensor([[1.0, 0.0]], device=DEVICE).expand(1024, 2))
_dv = env.sim_adapter.root_lin_vel_w[:, 0] - _v0
print(f"   0.2초 동안 속도 변화 평균 {_dv.mean():.3f} m/s  (τ/(1.4 m r)·0.2 = {0.2 / 0.28:.3f})")
check("액션이 물리에 반영됨", 0.6 < _dv.mean() < 0.8)

# 경기장 밖으로 내보내기 → terminated, 자동 리셋
_ids = torch.tensor([3, 5], device=DEVICE)
env.sim_adapter.write_root_state(_ids, torch.tensor([[6.0, 0.0, 0.2]], device=DEVICE).expand(2, 3),
                                 torch.tensor([[1.0, 0, 0, 0]], device=DEVICE).expand(2, 4), torch.zeros(2, 3, device=DEVICE), torch.zeros(2, 3, device=DEVICE))
obs, rew, terminated, time_outs, extras = env.step(_zero)
check("밖으로 나간 환경은 terminated (time_out 아님)", terminated[_ids].all() and not time_outs[_ids].any() and terminated.sum() == 2)
check("그 환경은 이미 리셋되어 있음: 길이 0, 위치 ±2", (env.episode_length_buf[_ids] == 0).all() and env.sim_adapter.root_pos_w[_ids, :2].abs().max() <= 2.0)
check("돌려준 obs 는 새 에피소드의 것 (직전 액션 칸이 0)", obs["critic"][_ids, 7:9].abs().sum() == 0)
check("terminated 만 있을 때는 terminal_obs 가 없다", "terminal_obs" not in extras)
check_close("종료 로그", extras["log"]["Episode_Termination/out_of_bounds"], 1.0)

# 시간 초과까지 돌리기
env = ManagerBasedRlEnv(BallEnvCfg(scene=SceneCfg(num_envs=1024)))
env.reset()
_zero = torch.zeros(1024, 2, device=DEVICE)
for _ in range(399):
    obs, rew, terminated, time_outs, extras = env.step(_zero)
_alive = env.episode_length_buf == 399           # 중간에 밖으로 나가 리셋된 적이 없는 환경
check("399 스텝째: 아직 시간 초과 없음", not time_outs.any())
print(f"   액션 0 으로 399 스텝을 버틴 환경: {int(_alive.sum())} / 1024  (나머지는 걷어차여 굴러가다 밖으로 나감)")
obs, rew, terminated, time_outs, extras = env.step(_zero)
check("400 스텝째: 버틴 환경은 전부 time_out", time_outs[_alive].all() and not time_outs[~_alive].any())
_pure = time_outs & ~terminated                   # 같은 스텝에 밖으로도 나갔다면 terminated 가 우선
check("terminal_obs: 순수 시간 초과 환경의 리셋 직전 critic 관측", torch.equal(extras["terminal_obs"]["env_ids"], _pure.nonzero().squeeze(-1))
      and extras["terminal_obs"]["obs"].shape == (int(_pure.sum()), 10) and _pure.sum() > 0)
check("자동 리셋됨", (env.episode_length_buf[_alive] == 0).all())
check("마지막 스텝의 보상은 리셋 전 상태로 계산됨 (0 이 아님)", (rew[_alive] > 0).all())
print(f"   액션 0 으로 8초: 에피소드 보상 합의 평균 = coarse {extras['log']['Episode_Reward/goal_tracking_coarse']:.3f}, fine {extras['log']['Episode_Reward/goal_tracking_fine']:.3f}")

# 같은 seed → 같은 결과
def _rollout(seed):
    e = ManagerBasedRlEnv(BallEnvCfg(seed=seed, scene=SceneCfg(num_envs=256)))
    o, _ = e.reset()
    g = torch.Generator(device=DEVICE).manual_seed(123)
    for _ in range(30):
        o, r, *_ = e.step(torch.randn(256, 2, device=DEVICE, generator=g))
    return o["critic"], r

_o1, _r1 = _rollout(0); _o2, _r2 = _rollout(0); _o3, _ = _rollout(1)
check_close("같은 seed 면 30 스텝 뒤 관측이 같음", _o1, _o2, atol=1e-5)
check("다른 seed 면 다름", (_o1 - _o3).abs().max() > 0.1)


### 속도 재기


In [ ]:
env = ManagerBasedRlEnv(BallEnvCfg())
env.reset()
_a = torch.zeros(env.num_envs, 2, device=DEVICE)
for _ in range(5):
    env.step(_a)
wp.synchronize(); _t = time.time()
for _ in range(100):
    env.step(_a)
wp.synchronize(); _dt = time.time() - _t
print(f"환경 {env.num_envs} 개 × 100 스텝: {_dt:.2f} 초  →  초당 {env.num_envs * 100 / _dt:,.0f} 환경 스텝 (물리 스텝으로는 ×{env.cfg.decimation})")


**실제 코드에서는** `step()` 에 다음이 더 있습니다. 이름을 알아 두면 진짜 코드를 읽을 때 길을 잃지 않습니다.

- `self.cache.invalidate()` — `SimAdapter` 의 캐시 무효화 (Part 3)
- `self.sensor_latency.*` — 센서 지연 모델. 물리 서브스텝마다 센서값을 테이프에 기록해 두고 관측에서 "몇 ms 전 값" 을 읽습니다
- `_compute_all_regime_masks()` — 보상 항의 regime 판정 (Part 8)
- `custom_managers` 의 `advance()` — `LocomotionManager`(보행 위상과 발 궤적), `ContactManager`(접촉 센서). 종료·보상보다 먼저 갱신됩니다
- `curriculum_manager.compute()` — 리셋 때 난이도 조정
- `extras["learning_valid"]` — 리셋 직후 워밍업 구간의 표본을 학습에서 빼는 마스크
- `_profile_step` — 단계별 시간 측정. 학습이 느릴 때 어디가 병목인지 볼 수 있습니다

그리고 World 의 환경은 세 클래스에 나뉘어 있습니다. `ManagerBasedEnv`(엔진과 무관한 공통 뼈대)와 `ManagerBasedRlEnv`(보상·종료·명령이 붙은 RL 환경)가 `diden-env` 패키지에 있고, 엔진에 닿는 부분(`create_scene`, `perform_physics_step`, `post_reset_sim_sync`)만 `diden-mjlab` 의 `MujocoWarpEnv` 가 구현합니다. 우리는 그 셋을 한 클래스에 합쳤습니다.


## Part 12 — 네트워크: `RunningNormalization` 과 `ActorCritic`

환경이 끝났으니 학습 알고리즘입니다. World 의 PPO 는 `diden-rl` 패키지에 있습니다. 문서는 이 패키지를 "frozen RL island" 라고 부르는데, 다른 패키지와 달리 거의 고치지 않고 통째로 가져다 쓰는 부분이라는 뜻입니다. 구조는 `rsl_rl` 계열과 같습니다.

```
OnPolicyRunner.learn()
  반복:
    1. 수집   obs → PPO.act() → env.step() → PPO.process_env_step()      × num_steps_per_env
    2. 반환값 PPO.compute_returns()   (RolloutStorage 가 GAE 계산)
    3. 갱신   PPO.update()            (미니배치로 여러 epoch)
```

### 관측 정규화

관측 항마다 크기가 다릅니다. 각속도는 ±10 rad/s 까지 가고 마찰계수는 0.3~1.0 입니다. 신경망은 입력 크기가 비슷할 때 잘 배우므로, 입구에서 항마다 평균을 빼고 표준편차로 나눕니다. 평균과 분산은 미리 알 수 없으니 **들어오는 관측으로 계속 갱신** 합니다(`diden_rl/modules/utils.py` 의 `RunningNormalization`).

지금까지 본 표본 수가 $n$, 평균 $\mu$, 분산 $\sigma^2$ 일 때 크기 $m$ 인 새 배치(평균 $\mu_b$, 분산 $\sigma_b^2$)를 합치면

$$\delta = \mu_b - \mu,\quad N = n + m,\qquad \mu' = \mu + \delta\,\frac{m}{N},\qquad \sigma'^2 = \frac{\sigma^2 n + \sigma_b^2 m + \delta^2\, n m / N}{N}$$

이 평균·분산은 정책의 일부입니다. 배포할 때 같이 가져가지 않으면 정책은 학습 때와 전혀 다른 크기의 입력을 받습니다. World 는 ONNX 를 만들 때 정규화를 그래프 안에 넣어 버립니다(Part 17). Core 가 관측에 아무 스케일도 곱하지 않는 이유입니다.

### 문제 12-1. `RunningNormalization`

`forward(x)` 는 **현재 통계로 정규화한 값** 을 돌려주고, 학습 모드(`self.training`)일 때만 그 뒤에 통계를 갱신합니다.


In [ ]:
class RunningNormalization(nn.Module):
    """(World: diden_rl/modules/utils.py)"""

    def __init__(self, num_inputs: int, epsilon: float = 1e-8):
        super().__init__()
        self.register_buffer("running_mean", torch.zeros(num_inputs))
        self.register_buffer("running_var", torch.ones(num_inputs))
        self.register_buffer("count", torch.ones(1))
        self.epsilon = epsilon

    @torch.no_grad()
    def update_stats(self, x: torch.Tensor) -> None:
        batch_mean, batch_var, batch_count = x.mean(0), x.var(0, unbiased=False), x.shape[0]
        delta = batch_mean - self.running_mean
        total = self.count + batch_count
        new_mean = self.running_mean + delta * batch_count / total
        new_var = (self.running_var * self.count + batch_var * batch_count
                   + delta.pow(2) * self.count * batch_count / total) / total
        self.running_mean.copy_(new_mean)
        self.running_var.copy_(new_var)
        self.count.copy_(total)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = (x - self.running_mean) / torch.sqrt(self.running_var + self.epsilon)
        if self.training:
            self.update_stats(x)
        return normalized


In [ ]:
torch.manual_seed(0)
_norm = RunningNormalization(3).to(DEVICE)
_data = torch.randn(50000, 3, device=DEVICE) * torch.tensor([0.1, 1.0, 10.0], device=DEVICE) + torch.tensor([5.0, 0.0, -3.0], device=DEVICE)
_first = _norm(_data[:1000])
check_close("첫 호출은 초기 통계(평균 0, 분산 1)로 정규화 → 입력 그대로", _first, _data[:1000], atol=1e-5)
for _i in range(1000, 50000, 1000):
    _norm(_data[_i:_i + 1000])
check_close("평균 수렴", _norm.running_mean, [5.0, 0.0, -3.0], atol=0.1)
check_close("표준편차 수렴", _norm.running_var.sqrt(), [0.1, 1.0, 10.0], rtol=0.03)
_out = _norm(_data)
check_close("정규화 결과는 평균 0, 표준편차 1", [_out.mean().item(), _out.std().item()], [0.0, 1.0], atol=0.03)
_norm.eval()
_c = _norm.count.clone()
_norm(_data)
check("eval 모드에서는 통계가 갱신되지 않음", torch.equal(_norm.count, _c))


### 정책은 분포를 낸다

PPO 의 정책은 액션 하나를 내지 않고 **가우시안 분포** 를 냅니다. 평균 $\mu(o)$ 는 신경망이 내고, 표준편차 $\sigma$ 는 관측과 무관한 학습 파라미터입니다. 학습 중에는 이 분포에서 뽑아(탐색) 쓰고, 배포할 때는 평균만 씁니다.

$$a \sim \mathcal N(\mu_\theta(o),\ \sigma^2), \qquad \log \pi(a \mid o) = \sum_i \Big[-\frac{(a_i - \mu_i)^2}{2\sigma_i^2} - \log \sigma_i - \tfrac12 \log 2\pi\Big]$$

가치 네트워크(critic)는 따로 있고, critic 관측을 받아 "지금부터 받을 할인 보상의 합" 을 숫자 하나로 냅니다.

### 문제 12-2. `ActorCritic`

- `actor`, `critic`: MLP. 은닉층 활성화는 ELU, 출력층에는 활성화가 없습니다. `create_mlp` 는 주어져 있습니다.
- `self.std`: 크기 `num_actions` 의 `nn.Parameter`, 초기값 `init_noise_std`.
- `update_distribution(actor_obs)`: 정규화 → actor → `torch.distributions.Normal(mean, std)` 를 `self.distribution` 에 저장. std 는 `min_std` 이상으로 자릅니다.
- `act(actor_obs, deterministic=False)`: 분포를 갱신하고 표본(또는 평균)을 돌려준다.
- `get_actions_log_prob(actions)`: 현재 분포에서의 로그 확률. 액션 차원에 대해 **합** 해서 `(N,)`.
- `evaluate(critic_obs)`: 정규화 → critic → `(N, 1)`.
- `action_mean`, `action_std`, `entropy`: 현재 분포의 것. `entropy` 는 차원에 대해 합.

World 의 초기화(`actor_critic.py`)를 따라 가중치는 직교 초기화하고, actor 의 **마지막 층은 0.01 배** 로 줄입니다. 처음에 평균이 거의 0 이 되어 탐색이 `std` 로만 정해지게 하기 위해서입니다.


In [ ]:
def create_mlp(input_dim: int, hidden_dims, output_dim: int) -> nn.Sequential:
    """(주어진 코드) ELU MLP."""
    layers, d = [], input_dim
    for h in hidden_dims:
        layers += [nn.Linear(d, h), nn.ELU()]
        d = h
    layers.append(nn.Linear(d, output_dim))
    return nn.Sequential(*layers)


def orthogonal_init(module: nn.Module, last_layer_scale: float) -> None:
    """(주어진 코드) 직교 초기화. 마지막 Linear 만 last_layer_scale 배."""
    linears = [m for m in module.modules() if isinstance(m, nn.Linear)]
    for i, m in enumerate(linears):
        nn.init.orthogonal_(m.weight, gain=last_layer_scale if i == len(linears) - 1 else math.sqrt(2))
        nn.init.zeros_(m.bias)


class ActorCritic(nn.Module):
    """(World: diden_rl/modules/actor_critic.py)"""

    def __init__(self, num_actor_obs: int, num_critic_obs: int, num_actions: int,
                 actor_hidden_dims=(128, 64), critic_hidden_dims=(128, 64), init_noise_std: float = 1.0, min_std: float = 0.01):
        super().__init__()
        self.min_std = min_std
        self.distribution = None
        self.actor_obs_normalizer = RunningNormalization(num_actor_obs)
        self.critic_obs_normalizer = RunningNormalization(num_critic_obs)
        self.actor = create_mlp(num_actor_obs, actor_hidden_dims, num_actions)
        self.critic = create_mlp(num_critic_obs, critic_hidden_dims, 1)
        orthogonal_init(self.actor, last_layer_scale=0.01)
        orthogonal_init(self.critic, last_layer_scale=1.0)
        self.std = nn.Parameter(init_noise_std * torch.ones(num_actions))

    def update_distribution(self, actor_obs: torch.Tensor) -> None:
        mean = self.actor(self.actor_obs_normalizer(actor_obs))
        std = self.std.clamp(min=self.min_std).expand_as(mean)
        self.distribution = torch.distributions.Normal(mean, std)

    def act(self, actor_obs: torch.Tensor, deterministic: bool = False) -> torch.Tensor:
        self.update_distribution(actor_obs)
        return self.distribution.mean if deterministic else self.distribution.sample()

    def get_actions_log_prob(self, actions: torch.Tensor) -> torch.Tensor:
        return self.distribution.log_prob(actions).sum(dim=-1)

    def evaluate(self, critic_obs: torch.Tensor) -> torch.Tensor:
        return self.critic(self.critic_obs_normalizer(critic_obs))

    @property
    def action_mean(self) -> torch.Tensor:
        return self.distribution.mean

    @property
    def action_std(self) -> torch.Tensor:
        return self.distribution.stddev

    @property
    def entropy(self) -> torch.Tensor:
        return self.distribution.entropy().sum(dim=-1)


In [ ]:
torch.manual_seed(0)
_ac = ActorCritic(9, 10, 2).to(DEVICE)
_o, _co = torch.randn(4096, 9, device=DEVICE), torch.randn(4096, 10, device=DEVICE)
_a = _ac.act(_o)
check("액션 (N, 2), 가치 (N, 1)", _a.shape == (4096, 2) and _ac.evaluate(_co).shape == (4096, 1))
check("처음에는 평균이 거의 0 (마지막 층 0.01 배)", _ac.action_mean.abs().max() < 0.1)
check_close("처음에는 표본의 표준편차 ≈ init_noise_std", _a.std().item(), 1.0, atol=0.05)
_lp = _ac.get_actions_log_prob(_a)
_want = (-(_a - _ac.action_mean) ** 2 / (2 * _ac.action_std ** 2) - _ac.action_std.log() - 0.5 * math.log(2 * math.pi)).sum(-1)
check_close("log_prob = 식 그대로, 차원에 대해 합", _lp, _want, atol=1e-4)
check_close("엔트로피 = Σ (0.5 + 0.5 log 2π + log σ)", _ac.entropy, torch.full((4096,), 2 * (0.5 + 0.5 * math.log(2 * math.pi))), atol=1e-4)
check_close("deterministic 이면 평균", _ac.act(_o, deterministic=True), _ac.action_mean, atol=0)
check("std 는 학습 파라미터", any(p is _ac.std for p in _ac.parameters()))
check("정규화기가 actor/critic 따로", _ac.actor_obs_normalizer.running_mean.shape == (9,) and _ac.critic_obs_normalizer.running_mean.shape == (10,))


**실제 코드에서는**: Spider 의 네트워크는 은닉층 `(512, 256, 64)`, 활성화 tanh, 층마다 LayerNorm, 초기 std 3.0 입니다(`diden_rl/configs/`). std 를 다루는 방식도 고정 / 학습 / 스케줄 / 적응형 네 가지가 있습니다(`std_type`). 그리고 actor 입력에 **상태 추정기의 출력** 이 덧붙습니다. 추정기는 actor 관측에서 몸통 선속도·발 접촉·발 높이·자석 힘 상태를 맞히도록 지도학습되는 작은 네트워크입니다(`diden_rl/modules/state_estimator.py`, `algorithms/ppo_estimator.py`).


## Part 13 — `RolloutStorage` 와 GAE

PPO 는 on-policy 입니다. **지금 정책** 으로 모은 데이터만 쓰고, 한 번 갱신하면 버립니다. 그 데이터를 담는 곳이 `RolloutStorage` 이고 모양은 `(num_steps_per_env, num_envs, ...)` 입니다. 환경 4096 개 × 24 스텝이면 한 번에 98,304 개의 전이(transition)가 쌓입니다.

### 이점(advantage)

정책 경사의 핵심은 "이 액션이 **평소보다** 얼마나 좋았는가" 입니다. 그것이 이점 $A_t$ 이고, 이점이 양수인 액션의 확률을 높이고 음수인 액션의 확률을 낮춥니다.

가장 단순한 추정은 한 스텝짜리 TD 오차입니다.

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

$V$ 가 정확하면 이것으로 충분하지만 학습 초반의 $V$ 는 엉터리입니다. 반대로 에피소드 끝까지의 실제 보상을 다 더하면 $V$ 에 의존하지 않는 대신 분산이 큽니다. **GAE** 는 둘을 $\lambda$ 로 섞습니다.

$$A_t = \delta_t + (\gamma\lambda)\,\delta_{t+1} + (\gamma\lambda)^2\,\delta_{t+2} + \cdots$$

뒤에서부터 거꾸로 계산하면 점화식 한 줄입니다. 에피소드가 끝난 스텝(`done`)에서는 다음 상태가 다른 에피소드이므로 이어 붙이지 않습니다.

```
advantage = 0
for step in 뒤에서부터:
    next_values = (마지막 스텝이면) last_values  (아니면) values[step + 1]
    not_done    = 1 − dones[step]
    delta       = rewards[step] + not_done · γ · next_values − values[step]
    advantage   = delta + not_done · γ · λ · advantage
    returns[step] = advantage + values[step]
```

`returns` 는 critic 의 학습 목표이고, `advantages = returns − values` 는 마지막에 평균 0, 표준편차 1 로 정규화합니다. World 의 `RolloutStorage.compute_returns` (`diden_rl/storage/rollout_storage.py`)가 이 코드입니다.

### 먼저 예상해 보기

$\lambda = 0$ 이면 $A_t$ 는 무엇이 되나요? $\lambda = 1$ 이고 $V \equiv 0$ 이면 `returns[t]` 는 무엇이 되나요?

### 문제 13-1. `RolloutStorage`

- `add_transitions(...)`: 현재 스텝 칸에 한 스텝 분량을 써넣고 `self.step` 을 1 늘린다. `rewards`, `dones`, `values`, `actions_log_prob` 는 `(N,)` 로 들어오며 `(N, 1)` 로 저장한다.
- `compute_returns(last_values, gamma, lam)`: 위 점화식.
- `mini_batch_generator(num_mini_batches, num_epochs)`: 주어져 있음. `(T, N, ...)` 를 `(T·N, ...)` 로 펴고 매 epoch 마다 섞어 `num_mini_batches` 조각으로 나눠 준다.


In [ ]:
class RolloutStorage:
    """(World: diden_rl/storage/rollout_storage.py)"""

    def __init__(self, num_envs: int, num_transitions_per_env: int, actor_obs_dim: int, critic_obs_dim: int, action_dim: int, device=DEVICE):
        T, N = num_transitions_per_env, num_envs
        self.num_envs, self.num_transitions_per_env, self.device = N, T, device
        self.actor_observations = torch.zeros(T, N, actor_obs_dim, device=device)
        self.critic_observations = torch.zeros(T, N, critic_obs_dim, device=device)
        self.actions = torch.zeros(T, N, action_dim, device=device)
        self.rewards = torch.zeros(T, N, 1, device=device)
        self.dones = torch.zeros(T, N, 1, device=device)
        self.values = torch.zeros(T, N, 1, device=device)
        self.actions_log_prob = torch.zeros(T, N, 1, device=device)
        self.mu = torch.zeros(T, N, action_dim, device=device)
        self.sigma = torch.zeros(T, N, action_dim, device=device)
        self.returns = torch.zeros(T, N, 1, device=device)
        self.advantages = torch.zeros(T, N, 1, device=device)
        self.step = 0

    def add_transitions(self, actor_obs, critic_obs, actions, rewards, dones, values, actions_log_prob, action_mean, action_sigma) -> None:
        assert self.step < self.num_transitions_per_env, "rollout 버퍼가 가득 찼다 — clear() 를 불렀는가?"
        s = self.step
        self.actor_observations[s] = actor_obs
        self.critic_observations[s] = critic_obs
        self.actions[s] = actions
        self.rewards[s] = rewards.view(-1, 1)
        self.dones[s] = dones.view(-1, 1).float()
        self.values[s] = values.view(-1, 1)
        self.actions_log_prob[s] = actions_log_prob.view(-1, 1)
        self.mu[s] = action_mean
        self.sigma[s] = action_sigma
        self.step += 1

    def clear(self) -> None:
        self.step = 0

    def compute_returns(self, last_values: torch.Tensor, gamma: float, lam: float) -> None:
        advantage = 0
        for step in reversed(range(self.num_transitions_per_env)):
            next_values = last_values if step == self.num_transitions_per_env - 1 else self.values[step + 1]
            next_is_not_terminal = 1.0 - self.dones[step]
            delta = self.rewards[step] + next_is_not_terminal * gamma * next_values - self.values[step]
            advantage = delta + next_is_not_terminal * gamma * lam * advantage
            self.returns[step] = advantage + self.values[step]
        self.advantages = self.returns - self.values
        self.advantages = (self.advantages - self.advantages.mean()) / (self.advantages.std() + 1e-8)

    def mini_batch_generator(self, num_mini_batches: int, num_epochs: int):
        batch_size = self.num_envs * self.num_transitions_per_env
        mini_batch_size = batch_size // num_mini_batches
        flat = lambda x: x.flatten(0, 1)
        fields_ = [flat(x) for x in (self.actor_observations, self.critic_observations, self.actions, self.values, self.advantages,
                                     self.returns, self.actions_log_prob, self.mu, self.sigma)]
        for _ in range(num_epochs):
            indices = torch.randperm(num_mini_batches * mini_batch_size, device=self.device)
            for i in range(num_mini_batches):
                idx = indices[i * mini_batch_size:(i + 1) * mini_batch_size]
                yield tuple(f[idx] for f in fields_)


In [ ]:
def _fill(storage, rewards, values, dones):
    """rewards/values/dones: (T, N) 리스트."""
    storage.clear()
    T, N = storage.num_transitions_per_env, storage.num_envs
    for t in range(T):
        storage.add_transitions(torch.zeros(N, 1, device=DEVICE), torch.zeros(N, 1, device=DEVICE), torch.zeros(N, 1, device=DEVICE),
                                torch.tensor(rewards[t], device=DEVICE), torch.tensor(dones[t], device=DEVICE), torch.tensor(values[t], device=DEVICE),
                                torch.zeros(N, device=DEVICE), torch.zeros(N, 1, device=DEVICE), torch.ones(N, 1, device=DEVICE))

_st = RolloutStorage(num_envs=2, num_transitions_per_env=3, actor_obs_dim=1, critic_obs_dim=1, action_dim=1)
# 환경 0: 끊김 없음.  환경 1: 두 번째 스텝에서 에피소드가 끝남.
_R = [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]
_V = [[0.5, 0.5], [0.4, 0.4], [0.3, 0.3]]
_D = [[False, False], [False, True], [False, False]]
_fill(_st, _R, _V, _D)
check("저장 모양 (T, N, 1)", _st.rewards.shape == (3, 2, 1) and _st.step == 3)
_last = torch.tensor([[0.2], [0.2]], device=DEVICE)
_g, _l = 0.9, 0.8
_st.compute_returns(_last, _g, _l)
# 손으로: 환경 0
_d2 = 3.0 + _g * 0.2 - 0.3; _d1 = 2.0 + _g * 0.3 - 0.4; _d0 = 1.0 + _g * 0.4 - 0.5
_A2 = _d2; _A1 = _d1 + _g * _l * _A2; _A0 = _d0 + _g * _l * _A1
check_close("환경 0 의 returns = A + V", _st.returns[:, 0, 0], [_A0 + 0.5, _A1 + 0.4, _A2 + 0.3], atol=1e-5)
# 환경 1: step 1 에서 done → 그 스텝은 다음 값을 보지 않고, 그 앞 스텝은 이어진다
_e1 = 2.0 - 0.4; _e0 = 1.0 + _g * 0.4 - 0.5 + _g * _l * _e1
check_close("환경 1: done 에서 끊김", _st.returns[:, 1, 0], [_e0 + 0.5, _e1 + 0.4, _A2 + 0.3], atol=1e-5)
check_close("advantages 는 정규화됨 (평균 0, 표준편차 1)", [_st.advantages.mean().item(), _st.advantages.std().item()], [0.0, 1.0], atol=1e-5)

_fill(_st, _R, [[0.0, 0.0]] * 3, [[False, False]] * 3)
_st.compute_returns(torch.zeros(2, 1, device=DEVICE), 1.0, 1.0)
check_close("γ=λ=1, V=0 이면 returns = 남은 보상의 합", _st.returns[:, 0, 0], [6.0, 5.0, 3.0])
_fill(_st, _R, _V, [[False, False]] * 3)
_st.compute_returns(_last, _g, 0.0)
check_close("λ=0 이면 returns − V = 한 스텝 TD 오차", (_st.returns - _st.values)[:, 0, 0], [_d0, _d1, _d2], atol=1e-5)

_batches = list(_st.mini_batch_generator(num_mini_batches=2, num_epochs=3))
check("미니배치: epoch 3 × 조각 2 = 6 개, 조각 크기 3", len(_batches) == 6 and _batches[0][0].shape == (3, 1) and len(_batches[0]) == 9)


**실제 코드에서는**: World 의 저장소에는 `valids` 마스크가 하나 더 있습니다. 리셋 직후 워밍업 구간(로봇이 자세를 잡는 동안)의 표본은 정책의 액션과 무관하므로 이점 정규화와 손실에서 제외합니다. 관측을 사전(dict) 그대로 저장하는 모드, RNN 용 은닉 상태 저장도 있습니다.


## Part 14 — `PPO`

정책을 고치는 가장 단순한 방법은 $\mathbb E[\log\pi(a \mid o)\,A]$ 를 키우는 것이지만, 한 번에 너무 많이 고치면 정책이 망가지고 망가진 정책으로 모은 데이터로는 회복하기 어렵습니다. PPO 는 "옛 정책에서 너무 멀어지지 않는 범위에서만" 고칩니다.

**대리 손실 (surrogate).** 새 정책과 옛 정책의 확률 비 $\rho = \pi_{\text{new}}(a \mid o) / \pi_{\text{old}}(a \mid o) = \exp(\log\pi_{\text{new}} - \log\pi_{\text{old}})$ 로

$$L^{\text{surr}} = \operatorname{mean}\Big[\max\big(-A\,\rho,\ -A\,\operatorname{clip}(\rho,\ 1-\epsilon,\ 1+\epsilon)\big)\Big]$$

$\rho$ 가 $[1-\epsilon, 1+\epsilon]$ 를 벗어나 **이득을 보는 방향** 으로는 기울기가 0 이 됩니다.

**가치 손실.** critic 도 같은 식으로 옛 값에서 $\pm\epsilon$ 넘게 움직이지 못하게 한 버전과 그냥 버전 중 큰 쪽을 씁니다.

$$V^{\text{clip}} = V_{\text{old}} + \operatorname{clip}(V - V_{\text{old}}, -\epsilon, \epsilon), \qquad L^{V} = \operatorname{mean}\big[\max\big((V - R)^2,\ (V^{\text{clip}} - R)^2\big)\big]$$

**전체 손실.** $L = L^{\text{surr}} + c_v L^{V} - c_e \cdot \operatorname{mean}[\text{entropy}]$. 엔트로피 항은 분포가 너무 빨리 좁아져 탐색이 멈추는 것을 막습니다.

**적응형 학습률.** 갱신할 때마다 옛 분포와 새 분포의 KL 발산을 재서, 목표값(`desired_kl = 0.01`)의 2 배를 넘으면 학습률을 1.5 로 나누고 절반에 못 미치면 1.5 를 곱합니다. 두 가우시안 사이의 KL 은

$$\mathrm{KL}(\text{old}\,\|\,\text{new}) = \sum_i \Big[\log\frac{\sigma_i}{\sigma_i^{\text{old}}} + \frac{(\sigma_i^{\text{old}})^2 + (\mu_i^{\text{old}} - \mu_i)^2}{2\sigma_i^2} - \frac12\Big]$$

네 가지 모두 World 의 `diden_rl/algorithms/ppo.py` 에 같은 이름의 메서드로 있습니다(`_compute_surrogate_loss`, `_compute_value_loss`, `_adaptive_learning_rate`).

### 문제 14-1. 손실 함수들


In [ ]:
def surrogate_loss(actions_log_prob: torch.Tensor, old_actions_log_prob: torch.Tensor, advantages: torch.Tensor, clip_param: float) -> torch.Tensor:
    """모두 (B,). 스칼라 손실을 돌려준다."""
    ratio = torch.exp(actions_log_prob - old_actions_log_prob)
    surrogate = -advantages * ratio
    surrogate_clipped = -advantages * torch.clamp(ratio, 1.0 - clip_param, 1.0 + clip_param)
    return torch.max(surrogate, surrogate_clipped).mean()


def value_loss(values: torch.Tensor, old_values: torch.Tensor, returns: torch.Tensor, clip_param: float) -> torch.Tensor:
    """모두 (B,)."""
    value_clipped = old_values + (values - old_values).clamp(-clip_param, clip_param)
    return torch.max((values - returns).pow(2), (value_clipped - returns).pow(2)).mean()


def gaussian_kl(mu: torch.Tensor, sigma: torch.Tensor, old_mu: torch.Tensor, old_sigma: torch.Tensor) -> torch.Tensor:
    """KL(old ‖ new). 입력 (B, A), 출력 (B,)."""
    return torch.sum(torch.log(sigma / old_sigma) + (old_sigma.square() + (old_mu - mu).square()) / (2.0 * sigma.square()) - 0.5, dim=-1)


In [ ]:
_adv = torch.tensor([1.0, -1.0, 2.0, -2.0])
_old = torch.zeros(4)
check_close("정책이 그대로면 (ρ=1) surrogate = −mean(A)", surrogate_loss(_old, _old, _adv, 0.2), -_adv.mean())
# A>0 인데 확률을 2배로: clip 이 걸려 1.2 까지만 인정.  A<0 인데 확률을 2배로: 벌은 clip 없이 다 받는다.
_new = torch.full((4,), math.log(2.0))
check_close("ρ=2: 좋은 액션은 1.2 까지만, 나쁜 액션은 2 그대로", surrogate_loss(_new, _old, _adv, 0.2), torch.tensor([-1.2, 2.0, -2.4, 4.0]).mean())
_new = torch.full((4,), math.log(0.5))
check_close("ρ=0.5: 좋은 액션을 줄이면 벌은 다 받고, 나쁜 액션을 줄인 이득은 0.8 까지만", surrogate_loss(_new, _old, _adv, 0.2), torch.tensor([-0.5, 0.8, -1.0, 1.6]).mean())
check_close("가치 손실: 조금 움직였으면 그냥 제곱오차", value_loss(torch.tensor([1.1]), torch.tensor([1.0]), torch.tensor([2.0]), 0.2), 0.81)
check_close("가치 손실: 목표 쪽으로 많이 움직여도 clip 된 쪽(더 큰 오차)을 쓴다", value_loss(torch.tensor([1.9]), torch.tensor([1.0]), torch.tensor([2.0]), 0.2), 0.64)
_g = torch.Generator().manual_seed(0)
_mu, _omu = torch.randn(8, 2, generator=_g), torch.randn(8, 2, generator=_g)
_sg, _osg = torch.rand(8, 2, generator=_g) + 0.5, torch.rand(8, 2, generator=_g) + 0.5
_want = torch.distributions.kl_divergence(torch.distributions.Normal(_omu, _osg), torch.distributions.Normal(_mu, _sg)).sum(-1)
check_close("KL = torch.distributions 와 일치", gaussian_kl(_mu, _sg, _omu, _osg), _want, atol=1e-5)
check_close("같은 분포면 KL = 0", gaussian_kl(_mu, _sg, _mu, _sg), torch.zeros(8), atol=1e-6)


### 시간 초과 처리

Part 9 와 11 에서 준비한 것을 여기서 씁니다. 시간 초과로 끝난 환경은 "계속했다면 받았을 가치" 를 그 스텝의 보상에 더해 줍니다.

$$r_t \leftarrow r_t + \gamma\, V(s_T^{\text{terminal}}) \qquad (\text{time\_out \& not terminated 인 환경만})$$

그리고 `done = terminated | time_outs` 로 저장해 GAE 가 에피소드 경계를 넘지 않게 합니다. World 의 `PPO._handle_timeout` 이 이 방식입니다.

### 문제 14-2. `PPO`

- `act(obs)`: 액션을 뽑고, 그때의 가치·로그확률·평균·표준편차·관측을 `self.transition` 에 보관한다. 전부 `detach()`.
- `process_env_step(rewards, terminated, time_outs, extras)`: 보상에 시간 초과 bootstrap 을 더하고 저장소에 넣는다.
- `compute_returns(last_critic_obs)`: 마지막 관측의 가치로 GAE.
- `update()`: 미니배치마다 분포·가치를 다시 계산 → KL 로 학습률 조정 → 손실 → 역전파 → 기울기 clip → 스텝. 끝나면 저장소를 비우고 평균 손실들을 돌려준다.

정규화 통계는 수집할 때만 갱신하고 `update()` 동안에는 고정합니다(World 의 `_frozen_obs_normalizer_stats`). 같은 표본이 epoch 수만큼 반복해서 통계에 들어가는 것을 막기 위해서입니다.


In [ ]:
@configclass
class PPOCfg:
    num_learning_epochs: int = 5
    num_mini_batches: int = 4
    clip_param: float = 0.2
    gamma: float = 0.99
    lam: float = 0.95
    value_loss_coef: float = 1.0
    entropy_coef: float = 0.005
    learning_rate: float = 1e-3
    desired_kl: float = 0.01
    max_grad_norm: float = 1.0


class PPO:
    """(World: diden_rl/algorithms/ppo.py)"""

    def __init__(self, actor_critic: ActorCritic, cfg: PPOCfg, device=DEVICE):
        self.actor_critic, self.cfg, self.device = actor_critic.to(device), cfg, device
        self.learning_rate = cfg.learning_rate
        self.optimizer = torch.optim.Adam(self.actor_critic.parameters(), lr=cfg.learning_rate)
        self.storage: RolloutStorage = None
        self.transition: dict = {}

    def init_storage(self, num_envs, num_transitions_per_env, actor_obs_dim, critic_obs_dim, action_dim) -> None:
        self.storage = RolloutStorage(num_envs, num_transitions_per_env, actor_obs_dim, critic_obs_dim, action_dim, self.device)

    def act(self, obs: dict) -> torch.Tensor:
        ac = self.actor_critic
        actions = ac.act(obs["actor"]).detach()
        self.transition = {
            "actor_obs": obs["actor"], "critic_obs": obs["critic"], "actions": actions,
            "values": ac.evaluate(obs["critic"]).detach(),
            "actions_log_prob": ac.get_actions_log_prob(actions).detach(),
            "action_mean": ac.action_mean.detach(), "action_sigma": ac.action_std.detach(),
        }
        return actions

    def process_env_step(self, rewards: torch.Tensor, terminated: torch.Tensor, time_outs: torch.Tensor, extras: dict) -> None:
        rewards = rewards.clone()
        terminal = extras.get("terminal_obs")
        if terminal is not None:
            with torch.no_grad():
                rewards[terminal["env_ids"]] += self.cfg.gamma * self.actor_critic.evaluate(terminal["obs"]).squeeze(-1)
        dones = terminated | time_outs
        t = self.transition
        self.storage.add_transitions(t["actor_obs"], t["critic_obs"], t["actions"], rewards, dones, t["values"],
                                     t["actions_log_prob"], t["action_mean"], t["action_sigma"])
        self.transition = {}

    def compute_returns(self, last_critic_obs: torch.Tensor) -> None:
        with torch.no_grad():
            last_values = self.actor_critic.evaluate(last_critic_obs)
        self.storage.compute_returns(last_values, self.cfg.gamma, self.cfg.lam)

    def update(self) -> dict:
        cfg, ac = self.cfg, self.actor_critic
        sums = {"surrogate": 0.0, "value": 0.0, "entropy": 0.0, "kl": 0.0}
        n_updates = 0
        ac.actor_obs_normalizer.eval(); ac.critic_obs_normalizer.eval()          # 통계 고정
        for (actor_obs, critic_obs, actions, old_values, advantages, returns, old_log_prob, old_mu, old_sigma) in \
                self.storage.mini_batch_generator(cfg.num_mini_batches, cfg.num_learning_epochs):
            ac.act(actor_obs)
            log_prob = ac.get_actions_log_prob(actions)
            values = ac.evaluate(critic_obs)
            mu, sigma, entropy = ac.action_mean, ac.action_std, ac.entropy

            with torch.no_grad():
                kl = gaussian_kl(mu, sigma, old_mu, old_sigma).mean()
                if kl > cfg.desired_kl * 2.0:
                    self.learning_rate = max(1e-5, self.learning_rate / 1.5)
                elif kl < cfg.desired_kl / 2.0 and kl > 0.0:
                    self.learning_rate = min(1e-2, self.learning_rate * 1.5)
                for group in self.optimizer.param_groups:
                    group["lr"] = self.learning_rate

            l_surr = surrogate_loss(log_prob, old_log_prob.squeeze(-1), advantages.squeeze(-1), cfg.clip_param)
            l_value = value_loss(values.squeeze(-1), old_values.squeeze(-1), returns.squeeze(-1), cfg.clip_param)
            loss = l_surr + cfg.value_loss_coef * l_value - cfg.entropy_coef * entropy.mean()

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(ac.parameters(), cfg.max_grad_norm)
            self.optimizer.step()
            sums["surrogate"] += l_surr.item(); sums["value"] += l_value.item()
            sums["entropy"] += entropy.mean().item(); sums["kl"] += kl.item()
            n_updates += 1
        ac.actor_obs_normalizer.train(); ac.critic_obs_normalizer.train()
        self.storage.clear()
        return {k: v / n_updates for k, v in sums.items()} | {"learning_rate": self.learning_rate}


In [ ]:
# 장난감 문제로 PPO 가 실제로 배우는지 확인한다: 관측 o ~ U(−1,1)², 보상 = −‖a − o‖². 최적 정책은 a = o.
torch.manual_seed(0)
_N, _T = 512, 8
_ac = ActorCritic(2, 2, 2, actor_hidden_dims=(32,), critic_hidden_dims=(32,))
_ppo = PPO(_ac, PPOCfg())
_ppo.init_storage(_N, _T, 2, 2, 2)
_o = torch.rand(_N, 2, device=DEVICE) * 2 - 1
_a = _ppo.act({"actor": _o, "critic": _o})
check("act: transition 에 필요한 것이 다 있고 기울기가 끊겨 있음", {"values", "actions_log_prob", "action_mean", "action_sigma"} <= set(_ppo.transition)
      and not _ppo.transition["values"].requires_grad and _a.shape == (_N, 2))

# 시간 초과 bootstrap
_term = torch.zeros(_N, dtype=torch.bool, device=DEVICE); _to = _term.clone(); _to[:3] = True
_tobs = torch.zeros(3, 2, device=DEVICE)
_v_term = _ac.evaluate(_tobs).detach().squeeze(-1)
_ppo.process_env_step(torch.ones(_N, device=DEVICE), _term, _to, {"terminal_obs": {"env_ids": torch.arange(3, device=DEVICE), "obs": _tobs}})
check_close("시간 초과 환경: 보상 += γ·V(terminal_obs)", _ppo.storage.rewards[0, :3, 0], 1.0 + 0.99 * _v_term, atol=1e-3)   # evaluate 를 부를 때마다 정규화 통계가 조금씩 움직이므로 느슨하게
check_close("나머지 환경: 보상 그대로", _ppo.storage.rewards[0, 3:, 0], torch.ones(_N - 3))
check("dones = terminated | time_outs", _ppo.storage.dones[0, :3, 0].all() and not _ppo.storage.dones[0, 3:, 0].any())
_ppo.storage.clear()

_errs = []
for _it in range(60):
    for _ in range(_T):
        _a = _ppo.act({"actor": _o, "critic": _o})
        _r = -(_a - _o).square().sum(-1)
        _done = torch.ones(_N, dtype=torch.bool, device=DEVICE)                 # 한 스텝짜리 에피소드
        _ppo.process_env_step(_r, _done, torch.zeros_like(_done), {})
        _o = torch.rand(_N, 2, device=DEVICE) * 2 - 1
    _ppo.compute_returns(_o)
    _info = _ppo.update()
    _errs.append((_ac.act(_o, deterministic=True) - _o).abs().mean().item())
print(f"   평균 |a − o|: 처음 {_errs[0]:.3f} → 끝 {_errs[-1]:.3f},  std: {_ac.std.data.tolist()},  lr: {_info['learning_rate']:.2e}")
check("정책이 a ≈ o 를 배움", _errs[-1] < 0.1 and _errs[-1] < 0.3 * _errs[0])
check("탐색 std 가 줄어듦", (_ac.std < 0.8).all())
check("update 뒤 저장소가 비워짐", _ppo.storage.step == 0)


**실제 코드에서는**: World 의 `PPO.update` 에는 선택 사항이 여럿 더 있습니다. 좌우 대칭 증강(`algorithms/symmetry.py` — 로봇을 좌우로 뒤집은 관측·액션을 데이터에 추가), 기울기 벌점(LCP — 관측이 조금 변할 때 액션이 급변하지 않게), actor 와 critic 의 학습률 분리, 그리고 추정기 손실(`ppo_estimator.py`). 뼈대는 위의 것과 같습니다.


## Part 15 — `OnPolicyRunner`: 학습

마지막 조각입니다. 수집 → 반환값 → 갱신을 반복하는 루프이고, World 에서는 `diden_rl/runners/on_policy_runner.py` 의 `learn()` → `_run_training_iteration()` → `_collect_experience()` 입니다. `dw train` 명령이 결국 부르는 것이 이 `learn()` 입니다(`tools/engines/mjwarp/train.py`).

```
obs = env.reset()
for it in range(num_iterations):
    # 1. 수집 (기울기 없이)
    for _ in range(num_steps_per_env):
        actions = alg.act(obs)
        obs, rewards, terminated, time_outs, extras = env.step(actions)
        alg.process_env_step(rewards, terminated, time_outs, extras)
    # 2. 반환값
    alg.compute_returns(obs["critic"])
    # 3. 갱신
    info = alg.update()
```

숫자 감각: 환경 4096 개 × 24 스텝 = 한 iteration 에 98,304 표본, 시뮬레이션 시간으로 환경마다 0.48 초입니다. 300 iteration 이면 약 2900 만 표본, 환경마다 144 초(에피소드 18 개)입니다.

### 문제 15-1. `learn`

기록용 코드(에피소드 보상 합·길이 집계)는 주어져 있습니다. 수집 루프와 갱신을 채웁니다. 수집은 `torch.no_grad()` 안에서 합니다.


In [ ]:
@configclass
class RunnerCfg:
    num_steps_per_env: int = 24
    actor_hidden_dims: tuple = (128, 64)
    critic_hidden_dims: tuple = (128, 64)
    init_noise_std: float = 1.0
    algorithm: PPOCfg = PPOCfg()


class OnPolicyRunner:
    """(World: diden_rl/runners/on_policy_runner.py)"""

    def __init__(self, env: ManagerBasedRlEnv, cfg: RunnerCfg):
        self.env, self.cfg = env, cfg
        dims = env.observation_manager.group_obs_dim
        self.actor_critic = ActorCritic(dims["actor"], dims["critic"], env.num_actions, cfg.actor_hidden_dims,
                                        cfg.critic_hidden_dims, cfg.init_noise_std).to(env.device)
        self.alg = PPO(self.actor_critic, cfg.algorithm, env.device)
        self.alg.init_storage(env.num_envs, cfg.num_steps_per_env, dims["actor"], dims["critic"], env.num_actions)
        self.history: list[dict] = []
        self.current_iteration = 0
        self._obs = None

    def learn(self, num_iterations: int, log_interval: int = 20) -> list[dict]:
        env, alg = self.env, self.alg
        if self._obs is None:
            self._obs, _ = env.reset()
        obs = self._obs
        ep_return = torch.zeros(env.num_envs, device=env.device)
        ep_length = torch.zeros(env.num_envs, device=env.device)
        for _ in range(num_iterations):
            t0 = time.time()
            done_returns, done_lengths, step_rewards, last_log = [], [], [], {}
            with torch.no_grad():
                for _ in range(self.cfg.num_steps_per_env):
                    actions = alg.act(obs)
                    obs, rewards, terminated, time_outs, extras = env.step(actions)
                    alg.process_env_step(rewards, terminated, time_outs, extras)
                    # ── 기록 (주어진 코드) ──
                    dones = terminated | time_outs
                    ep_return += rewards
                    ep_length += 1
                    step_rewards.append(rewards.mean().item())
                    if dones.any():
                        done_returns.append(ep_return[dones].clone()); done_lengths.append(ep_length[dones].clone())
                        ep_return[dones] = 0.0; ep_length[dones] = 0.0
                        last_log = dict(extras.get("log", {}))
            t_collect = time.time() - t0
            alg.compute_returns(obs["critic"])
            info = alg.update()
            self.current_iteration += 1
            rec = {"iteration": self.current_iteration, "mean_step_reward": float(np.mean(step_rewards)),
                   "mean_episode_return": torch.cat(done_returns).mean().item() if done_returns else float("nan"),
                   "mean_episode_length": torch.cat(done_lengths).mean().item() if done_lengths else float("nan"),
                   "action_std": self.actor_critic.std.mean().item(), "collect_s": t_collect, "update_s": time.time() - t0 - t_collect,
                   **info, **last_log}
            self.history.append(rec)
            if self.current_iteration % log_interval == 0 or self.current_iteration == 1:
                fps = env.num_envs * self.cfg.num_steps_per_env / (time.time() - t0)
                print(f"it {rec['iteration']:4d} | ep return {rec['mean_episode_return']:6.2f} | ep len {rec['mean_episode_length']:5.0f} | "
                      f"std {rec['action_std']:.2f} | kl {rec['kl']:.4f} | lr {rec['learning_rate']:.1e} | {fps:,.0f} steps/s")
        self._obs = obs
        return self.history


In [ ]:
# 짧게 돌려 배선이 맞는지만 확인한다 (환경 512 개, 3 iteration)
_env = ManagerBasedRlEnv(BallEnvCfg(scene=SceneCfg(num_envs=512)))
_runner = OnPolicyRunner(_env, RunnerCfg())
_w0 = _runner.actor_critic.actor[0].weight.clone()
_hist = _runner.learn(3, log_interval=1)
check("3 iteration 기록", len(_hist) == 3 and _hist[-1]["iteration"] == 3)
check("가중치가 갱신됨", not torch.equal(_w0, _runner.actor_critic.actor[0].weight))
check("저장소가 비워짐", _runner.alg.storage.step == 0)
check("정규화 통계가 수집 중에 갱신됨 (표본 수 = 512 × 24 × 3 + 1)", abs(_runner.actor_critic.actor_obs_normalizer.count.item() - (512 * 24 * 3 + 1)) < 1)
check("손실이 유한", all(math.isfinite(_hist[-1][k]) for k in ("surrogate", "value", "entropy", "kl")))


### 학습

환경 4096 개로 300 iteration 을 돌립니다. RTX 5070 기준 2~3 분입니다.

### 먼저 예상해 보기

- 아무것도 배우지 않은 정책(액션 0)의 에피소드 보상 합은 Part 11 에서 본 대로 coarse ≈ 1.9, fine ≈ 0.1, 합쳐서 2 쯤입니다. 완벽한 정책이라면 얼마쯤일까요? (목적지 위에 가만히 있으면 스텝마다 `(1 + 2)·0.02`, 에피소드는 400 스텝)
- 탐색 표준편차 `std` 는 학습이 진행되면서 어떻게 변할까요?


In [ ]:
env = ManagerBasedRlEnv(BallEnvCfg())
runner = OnPolicyRunner(env, RunnerCfg())
_t0 = time.time()
history = runner.learn(300)
print(f"학습 시간 {time.time() - _t0:.0f} 초")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
_it = [h["iteration"] for h in history]
axes[0].plot(_it, [h["mean_episode_return"] for h in history]); axes[0].set_title("episode return")
axes[1].plot(_it, [h.get("Episode_Reward/goal_tracking_coarse", np.nan) for h in history], label="coarse")
axes[1].plot(_it, [h.get("Episode_Reward/goal_tracking_fine", np.nan) for h in history], label="fine"); axes[1].legend(); axes[1].set_title("reward terms (episode sum)")
axes[2].plot(_it, [h.get("Episode_Termination/out_of_bounds", np.nan) for h in history]); axes[2].set_title("out_of_bounds fraction")
axes[3].plot(_it, [h["action_std"] for h in history]); axes[3].set_title("action std")
for ax in axes:
    ax.set_xlabel("iteration"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
_last = history[-20:]
_ret = float(np.nanmean([h["mean_episode_return"] for h in _last]))
print(f"   마지막 20 iteration 평균 에피소드 보상 합: {_ret:.2f}  (액션 0 일 때 약 2)")
check("에피소드 보상 합이 크게 늘었음 (> 10)", _ret > 10.0)
check("경기장 밖으로 나가는 비율이 줄었음 (< 5 %)", float(np.nanmean([h.get("Episode_Termination/out_of_bounds", np.nan) for h in _last])) < 0.05)
check("탐색 std 가 줄었음", history[-1]["action_std"] < 0.7)


그래프 읽기:

- **coarse 가 먼저, fine 이 나중에** 오릅니다. 정책은 먼저 "목적지 쪽으로 간다" 를 배우고, 그다음에 "정확히 멈춘다" 를 배웁니다. σ 가 넓은 커널을 같이 둔 이유입니다. fine 만 있으면 목적지에서 1 m 떨어진 곳과 3 m 떨어진 곳의 보상이 둘 다 0 이라 처음에 배울 신호가 없습니다.
- **out_of_bounds** 는 초반에 잠깐 올랐다가 0 으로 떨어집니다. 토크를 마구 쓰면 밖으로 나간다는 것을 먼저 겪습니다.
- **std** 는 서서히 줄어듭니다. 엔트로피 보너스가 이것을 늦추고, 이점이 뚜렷해질수록 줄어드는 쪽이 이깁니다.

**실제 코드에서는**: Spider 의 설정은 스텝 32 개, epoch 5, 미니배치 4, 학습률 2.5e-4(적응형, KL 0.01), γ 0.99, λ 0.95 입니다(`diden_rl/configs/`). 커리큘럼이 iteration 10000~15000 구간까지 걸쳐 있을 만큼 오래 돌립니다. 한 iteration 의 표본 수는 4096 × 32 = 131,072 개로 우리와 비슷하지만 iteration 수가 수십 배입니다. 그래서 학습을 원격 GPU 에 뿌리는 `dw fleet`(`tools/`)과 W&B·MLflow 기록(`diden_rl`)이 붙어 있습니다.


## Part 16 — 평가와 영상

학습 곡선이 올랐다고 정책이 좋은 것은 아닙니다. 보상은 우리가 임의로 정한 숫자이고, 원하는 것은 "목적지에 가서 멈춘다" 입니다. 그것을 직접 재는 지표를 따로 만듭니다.

**성공** 을 이렇게 정의합니다: 목적지가 주어지고 3.8 초 뒤에, 공이 목적지 표시(반지름 0.25 m) 안에 있고 속력이 0.3 m/s 미만이다.

평가용 환경은 학습용과 설정이 조금 다릅니다.

- 목적지가 정확히 4 초마다 바뀝니다(`resampling_time_range=(4, 4)`). 에피소드 8 초 동안 목적지 2 개.
- 걷어차기를 끕니다(`events.push_ball = None`). World 에서도 term 을 끄는 방법이 `None` 대입입니다.
- 관측 노이즈는 그대로 둡니다. 실제 센서에도 노이즈가 있습니다.
- 액션은 분포의 **평균** 을 씁니다(`deterministic=True`).

### 문제 16-1. 성공률 재기

`policy(obs_dict) -> actions` 형태의 함수를 받아 성공률을 잽니다. 정책이 신경망이든 손으로 짠 제어기든 같은 함수로 평가할 수 있게 합니다.

- 190 번째 스텝(3.8 초) 직후와 390 번째 스텝(7.8 초) 직후에 성공 여부를 판정한다.
- 그 사이에 한 번이라도 `terminated` 된 환경은 실패로 친다.
- 돌려주는 값: `{"success_1": 첫 목적지 성공률, "success_2": 둘째 목적지 성공률, "terminated": 중간에 밖으로 나간 비율, "final_distance": 판정 시점 거리의 평균}`


In [ ]:
def make_eval_cfg(num_envs: int = 2048, seed: int = 100, friction_range: tuple = (0.3, 1.0)) -> BallEnvCfg:
    """(주어진 코드) 평가용 설정."""
    cfg = BallEnvCfg(seed=seed, scene=SceneCfg(num_envs=num_envs))
    cfg.commands.goal.resampling_time_range = (4.0, 4.0)
    cfg.events.push_ball = None
    cfg.events.randomize_friction.params["friction_range"] = friction_range
    return cfg


def is_success(env, distance_threshold: float = 0.25, speed_threshold: float = 0.3) -> torch.Tensor:
    """(N,) bool: 목적지 안에 있고 거의 멈춰 있는가."""
    speed = env.sim_adapter.root_lin_vel_w[:, :2].norm(dim=-1)
    return (goal_distance(env) < distance_threshold) & (speed < speed_threshold)


@torch.no_grad()
def evaluate_policy(policy: Callable, cfg: BallEnvCfg = None) -> dict:
    env = ManagerBasedRlEnv(cfg if cfg is not None else make_eval_cfg())
    obs, _ = env.reset()
    ever_terminated = torch.zeros(env.num_envs, dtype=torch.bool, device=env.device)
    result, distances = {}, []
    for step in range(1, 391):
        obs, _, terminated, _, _ = env.step(policy(obs))
        ever_terminated |= terminated
        if step in (190, 390):
            ok = is_success(env) & ~ever_terminated
            result["success_1" if step == 190 else "success_2"] = ok.float().mean().item()
            distances.append(goal_distance(env)[~ever_terminated])
    result["terminated"] = ever_terminated.float().mean().item()
    result["final_distance"] = torch.cat(distances).mean().item()
    return result


In [ ]:
_r = evaluate_policy(lambda obs: torch.zeros(obs["actor"].shape[0], 2, device=DEVICE), make_eval_cfg(num_envs=512))
print("   액션 0:", {k: round(v, 3) for k, v in _r.items()})
check("결과 키", set(_r) == {"success_1", "success_2", "terminated", "final_distance"})
check("아무것도 안 하면 성공률은 거의 0", _r["success_1"] < 0.05 and _r["success_2"] < 0.05)


### 비교 대상: 손으로 짠 PD 제어기

학습된 정책이 얼마나 좋은지 알려면 기준이 필요합니다. 섹터 0 Part 6 의 PD 를 여기에 그대로 씁니다. 목적지까지의 상대 위치가 "오차", 속도가 "오차의 변화율" 입니다.

$$\mathbf a = k_p \cdot (\text{목적지} - \text{공}) - k_d \cdot \mathbf v$$

관측 벡터에서 필요한 칸을 직접 잘라 씁니다. actor 관측의 배치는 `[선속도 2 | 각속도 3 | 목적지 상대 위치 2 | 직전 액션 2]` 입니다.

### 먼저 예상해 보기

PD 제어기는 마찰이 1.0 인 바닥과 0.3 인 바닥에서 똑같이 잘할까요? 게인을 한 벌만 쓸 수 있다면 어느 쪽에 맞춰야 할까요?


In [ ]:
def pd_policy(kp: float, kd: float) -> Callable:
    def policy(obs: dict) -> torch.Tensor:
        o = obs["actor"]
        return kp * o[:, 5:7] - kd * o[:, 0:2]
    return policy


for _kp, _kd in [(1.0, 1.0), (2.0, 1.5), (4.0, 2.0)]:
    _row = [f"kp={_kp} kd={_kd}:"]
    for _name, _fr in (("마찰 0.3~1.0", (0.3, 1.0)), ("마찰 0.3 만", (0.3, 0.3001)), ("마찰 1.0 만", (0.9999, 1.0))):
        _r = evaluate_policy(pd_policy(_kp, _kd), make_eval_cfg(num_envs=1024, friction_range=_fr))
        _row.append(f"{_name} → 성공 {(_r['success_1'] + _r['success_2']) / 2:.2f}, 이탈 {_r['terminated']:.2f}")
    print("  ".join(_row))


### 학습된 정책 평가


In [ ]:
def trained_policy(actor_critic: ActorCritic) -> Callable:
    actor_critic.eval()                                      # 정규화 통계 고정
    return lambda obs: actor_critic.act(obs["actor"], deterministic=True)


for _name, _fr in (("마찰 0.3~1.0", (0.3, 1.0)), ("마찰 0.3 만", (0.3, 0.3001)), ("마찰 1.0 만", (0.9999, 1.0))):
    _r = evaluate_policy(trained_policy(runner.actor_critic), make_eval_cfg(friction_range=_fr))
    print(f"학습된 정책, {_name}: 성공 {(_r['success_1'] + _r['success_2']) / 2:.3f}  이탈 {_r['terminated']:.3f}  판정 시점 거리 {_r['final_distance']:.3f} m")
eval_result = evaluate_policy(trained_policy(runner.actor_critic))


In [ ]:
check(f"첫 목적지 성공률 {eval_result['success_1']:.3f} > 0.9", eval_result["success_1"] > 0.9)
check(f"둘째 목적지 성공률 {eval_result['success_2']:.3f} > 0.9", eval_result["success_2"] > 0.9)
check("경기장 이탈 < 1 %", eval_result["terminated"] < 0.01)


### 영상으로 보기

GPU 의 world 하나를 골라 그 상태를 CPU 의 `MjData` 에 복사하고 섹터 0 의 `mujoco.Renderer` 로 그립니다. World 의 `dw play` 도 같은 방식입니다(`diden_mjlab/env/native_visualizer.py`). 아래는 주어진 코드입니다.


In [ ]:
@torch.no_grad()
def record_policy(policy: Callable, cfg: BallEnvCfg, env_ids=(0, 1, 2, 3), steps: int = 400, every: int = 2, size=(360, 360)):
    """env_ids 환경들을 각각 영상으로 만든다. 프레임 목록의 리스트를 돌려준다."""
    env = ManagerBasedRlEnv(cfg)
    obs, _ = env.reset()
    mjm_, mjd_ = env.sim.mjm, mujoco.MjData(env.sim.mjm)
    frames = [[] for _ in env_ids]
    cam = make_camera(lookat=(0, 0, 0), distance=8.5, azimuth=90, elevation=-89)        # 위에서 내려다보기
    with mujoco.Renderer(mjm_, size[1], size[0]) as renderer:
        for step in range(steps):
            obs, *_ = env.step(policy(obs))
            if step % every:
                continue
            qpos, mocap = env.sim.qpos[list(env_ids)].cpu().numpy(), env.sim.mocap_pos[list(env_ids)].cpu().numpy()
            for k in range(len(env_ids)):
                mjd_.qpos[:] = qpos[k]
                mjd_.mocap_pos[:] = mocap[k]
                mujoco.mj_forward(mjm_, mjd_)
                renderer.update_scene(mjd_, camera=cam)
                frames[k].append(renderer.render())
    return frames


_cfg = make_eval_cfg(num_envs=16, seed=7)
print("학습된 정책 (환경 4개, 위에서 본 모습. 초록 원이 목적지이고 4초에 한 번 바뀝니다)")
media.show_videos(record_policy(trained_policy(runner.actor_critic), _cfg), fps=25, columns=4)
print("PD 제어기 kp=2, kd=1.5 (같은 seed → 같은 시작 위치와 목적지)")
media.show_videos(record_policy(pd_policy(2.0, 1.5), _cfg), fps=25, columns=4)


**부숴 보기** (각각 설정 한두 줄만 바꿔 다시 학습하고, 실행 전에 결과를 예상해 봅니다)

1. `RewardsCfg` 에서 `goal_tracking_coarse = None` 으로 끄면? fine 만으로 학습이 될까요?
2. 두 추종 항의 `weight` 를 음수 거리(`-d`)로 바꾸고 mul 벌점을 add 벌점으로 바꾸면(Part 8 의 (가) 설계) `out_of_bounds` 비율은 어떻게 될까요?
3. `TerminationsCfg` 에서 `time_out` 의 `time_out=True` 를 `False` 로 바꾸면(시간 초과를 실패로 취급) 가치 손실 곡선이 어떻게 달라질까요?
4. `randomize_friction` 의 범위를 `(1.0, 1.0)` 으로 고정해 학습한 정책을 마찰 0.3 에서 평가하면? 반대로 `(0.1, 1.0)` 으로 넓히면?
5. `ActionsCfg` 의 `scale` 을 5.0 으로 키우면? `decimation` 을 1 로(정책 200 Hz) 또는 20 으로(정책 10 Hz) 바꾸면?
6. actor 에서 `ball_lin_vel` 을 빼면(속도를 직접 못 봄) 정책은 멈출 수 있을까요? 각속도만으로 속도를 알 수 있는 경우와 없는 경우는?

4번이 도메인 랜덤화가 왜 필요한지, 6번이 Spider 에 상태 추정기가 왜 있는지를 직접 보여 줍니다.


## Part 17 — ONNX 로 내보내고 "Core 처럼" 실행하기

학습된 정책은 PyTorch 객체입니다. 로봇의 제어 PC 에서 도는 것은 C++ 로 된 DIDEN_Core 이고, 거기에는 PyTorch 가 없습니다. 둘 사이를 잇는 것이 **ONNX 파일 하나** 입니다. World 의 설계 문서는 이 경계를 **SEAM 3** 이라고 부릅니다.

파일에는 두 가지가 들어갑니다.

1. **계산 그래프**: 관측 벡터 → 액션 벡터. 관측 정규화가 그래프 **안에** 들어 있습니다. World 는 추정기까지 합쳐 `actor(normalize(cat(obs, estimator(obs))))` 를 한 그래프로 만듭니다(`tools/deploy/export_merged_onnx.py` 의 `MergedActorEstimator`).
2. **메타데이터 `policy_config`** (YAML 문자열): 이 그래프를 올바로 쓰는 데 필요한 약속 전부. 관측 항의 이름·순서·차원, 액션 스케일과 기준값, PD 게인, `decimation` 과 `physics_dt` 등. 학습에 쓴 환경에서 **자동으로** 뽑아 씁니다(`diden_env/deploy/serialize.py` 의 `build_policy_config`).

Core 는 정책을 올릴 때 먼저 이 설정을 읽고, 거기 적힌 순서대로 관측을 조립하고, 차원이 그래프 입력과 맞지 않으면 실행을 거부합니다(`rl_controller/policy.cpp`). 관측 순서나 액션 스케일을 사람이 양쪽에 따로 적어 맞추면 생기는 사고를 막기 위한 구조입니다.

**두 레포가 어긋나 보이는 곳 하나.** World 의 `policy_schema.py` 주석은 "Core 가 ONNX 의 `metadata_props` 에서 `policy_config` 를 읽는다" 고 적고 있습니다. 그런데 이 레포의 서브모듈이 가리키는 Core 커밋(`386bae3a8`)의 `policy.h` / `policy.cpp` 는 **ONNX 옆에 놓인 같은 이름의 `.yaml` 파일** 에서만 설정을 읽고, 없으면 로드에 실패합니다. 내용(YAML)은 같고 전달 경로가 다릅니다. 어느 쪽이 최신 약속인지는 코드만으로는 알 수 없으니 팀에 확인할 질문입니다(`QUESTION_TREE.md` Q4). 이 노트북에서는 World 의 방식(메타데이터에 넣기)을 따릅니다.

### 문제 17-1. 정규화를 품은 actor


In [ ]:
class MergedActor(nn.Module):
    """정규화 + actor 를 한 모듈로. 입력: 날것의 관측 (B, obs_dim), 출력: raw 액션의 평균 (B, action_dim)."""

    def __init__(self, actor_critic: ActorCritic):
        super().__init__()
        norm = actor_critic.actor_obs_normalizer
        self.register_buffer("mean", norm.running_mean.clone())
        self.register_buffer("var", norm.running_var.clone())
        self.epsilon = norm.epsilon
        self.actor = copy.deepcopy(actor_critic.actor)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.actor((obs - self.mean) / torch.sqrt(self.var + self.epsilon))


In [ ]:
_merged = MergedActor(runner.actor_critic).to(DEVICE).eval()
_o = torch.randn(64, 9, device=DEVICE) * 3
runner.actor_critic.eval()
check_close("MergedActor(obs) == 학습 때의 평균 액션", _merged(_o), runner.actor_critic.act(_o, deterministic=True), atol=1e-5)
check("탐색 std 는 들어 있지 않음 (배포에서는 평균만 쓴다)", not any("std" in n for n, _ in _merged.named_parameters()))


### 문제 17-2. `policy_config` 만들기

살아 있는 환경 객체에서 다음 사전을 만듭니다. 키 이름은 World 의 스키마(`diden_env/deploy/schema.py`)를 따랐습니다.

```yaml
rl:
  observations:            # actor 그룹의 term 을 정의 순서대로
    - {name: ball_lin_vel, dim: 2}
    - ...
  actions:                 # action term 을 정의 순서대로
    - {name: torque, dim: 2}
  control:
    action_scale: 1.0
    action_clip: 2.0
  decimation: 4
  physics_dt: 0.005
```

값은 전부 `env` 에서 읽습니다: `env.observation_manager.group_term_dims["actor"]`, `env.action_manager._terms`, 각 term 의 `cfg`, `env.cfg.decimation`, `env.physics_dt`.


In [ ]:
def build_policy_config(env: ManagerBasedRlEnv) -> dict:
    obs_terms = [{"name": n, "dim": int(d)} for n, d in env.observation_manager.group_term_dims["actor"].items()]
    act_terms = [{"name": n, "dim": int(t.action_dim)} for n, t in env.action_manager._terms.items()]
    term_cfg = next(iter(env.action_manager._terms.values())).cfg
    return {"rl": {
        "observations": obs_terms,
        "actions": act_terms,
        "control": {"action_scale": float(term_cfg.scale), "action_clip": None if term_cfg.clip is None else float(term_cfg.clip)},
        "decimation": int(env.cfg.decimation),
        "physics_dt": float(env.physics_dt),
    }}


In [ ]:
_pc = build_policy_config(env)
check("관측 항 순서와 차원", _pc["rl"]["observations"] == [{"name": "ball_lin_vel", "dim": 2}, {"name": "ball_ang_vel", "dim": 3},
                                                      {"name": "goal_rel_pos", "dim": 2}, {"name": "last_action", "dim": 2}])
check("액션", _pc["rl"]["actions"] == [{"name": "torque", "dim": 2}] and _pc["rl"]["control"] == {"action_scale": 1.0, "action_clip": 2.0})
check("주기: decimation × physics_dt = 0.02 s", _pc["rl"]["decimation"] == 4 and _pc["rl"]["physics_dt"] == 0.005)


### 문제 17-3. 내보내기

1. `torch.onnx.export(model, dummy, path, **ONNX_EXPORT_CONFIG)` — 설정은 World 의 것(`diden_env/deploy/policy_schema.py`)을 그대로 옮겼습니다. opset 11, 입력 이름 `input`, 출력 이름 `output`, 배치 축 가변.
2. `onnx.load` 로 다시 열어 `metadata_props` 에 두 항목을 추가한다: `model_kind` 와 `policy_config`(YAML 문자열).
3. `onnx.save`.

<details><summary>힌트</summary>

```python
proto = onnx.load(path)
proto.metadata_props.append(onnx.StringStringEntryProto(key=..., value=...))
onnx.save(proto, path)
```
YAML 문자열은 `yaml.safe_dump(config, sort_keys=False)` 로 만듭니다. `sort_keys=False` 를 빼면 키가 알파벳순으로 정렬됩니다(목록의 순서는 어느 쪽이든 유지됩니다).
</details>


In [ ]:
import warnings
import onnx
import onnxruntime as ort
import yaml

POLICY_CONFIG_METADATA_KEY = "policy_config"
MODEL_KIND_METADATA_KEY = "model_kind"
MODEL_KIND = "merged_actor"                       # World: "merged_actor_estimator"
ONNX_EXPORT_CONFIG = {
    "export_params": True, "opset_version": 11, "do_constant_folding": True,
    "input_names": ["input"], "output_names": ["output"],
    "dynamic_axes": {"input": {0: "batch"}, "output": {0: "batch"}},
    "dynamo": False,
}


def export_policy(actor_critic: ActorCritic, env: ManagerBasedRlEnv, path: str) -> dict:
    """ONNX 파일을 쓰고, 넣은 policy_config 사전을 돌려준다."""
    model = MergedActor(actor_critic).cpu().eval()
    config = build_policy_config(env)
    in_dim = sum(t["dim"] for t in config["rl"]["observations"])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.onnx.export(model, torch.zeros(1, in_dim), path, **ONNX_EXPORT_CONFIG)
    proto = onnx.load(path)
    proto.metadata_props.append(onnx.StringStringEntryProto(key=MODEL_KIND_METADATA_KEY, value=MODEL_KIND))
    proto.metadata_props.append(onnx.StringStringEntryProto(key=POLICY_CONFIG_METADATA_KEY, value=yaml.safe_dump(config, sort_keys=False)))
    onnx.save(proto, path)
    return config


In [ ]:
OUT_DIR = Path("_out"); OUT_DIR.mkdir(exist_ok=True)
ONNX_PATH = str(OUT_DIR / "ball_policy.onnx")
export_policy(runner.actor_critic, env, ONNX_PATH)
_sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
_meta = _sess.get_modelmeta().custom_metadata_map
check("입출력 이름과 모양", _sess.get_inputs()[0].name == "input" and _sess.get_inputs()[0].shape[1] == 9 and _sess.get_outputs()[0].shape[1] == 2)
check("메타데이터 두 항목", _meta.get("model_kind") == "merged_actor" and "policy_config" in _meta)
check("policy_config 를 다시 읽으면 원래 사전", yaml.safe_load(_meta["policy_config"]) == build_policy_config(env))
_x = np.random.default_rng(0).normal(size=(32, 9)).astype(np.float32) * 3
_y = _sess.run(None, {"input": _x})[0]
runner.actor_critic.eval()
check_close("ONNX 출력 == PyTorch 출력 (World 도 내보낼 때 이 검사를 한다)", _y, runner.actor_critic.act(torch.tensor(_x, device=DEVICE), deterministic=True), atol=1e-4)
print(f"   {ONNX_PATH}: {Path(ONNX_PATH).stat().st_size / 1024:.1f} KB")
print(_meta["policy_config"])


### "Core 처럼" 실행하기

이제 PyTorch 도, 우리가 만든 환경 클래스도 쓰지 않고 **ONNX 파일 하나와 MuJoCo 만으로** 공을 굴립니다. DIDEN_Core 의 RL 제어기(`applications/controller/runner/rl_controller/`)가 하는 일을 파이썬으로 줄인 것입니다.

| Core 가 하는 일 | 파일 | 여기서 |
|---|---|---|
| 정책 설정(YAML)을 읽는다. 관측·액션 차원 합이 그래프와 다르면 거부 | `policy.cpp`, `rl_controller.cpp` | `__init__` |
| YAML 에 적힌 **순서대로** 관측 항을 계산해 이어 붙인다. 이름 → 함수의 표가 있다 | `observation.cpp` | `OBS_TERMS`, `_build_obs` |
| 정책 주기 = `decimation × physics_dt`. 러너 스레드는 1 kHz 로 깨어나 시계를 보고, 주기가 찼을 때만 정책을 한 스텝 돌린다 | `runner/thread.cpp` | `tick` 의 카운터 (우리는 시계 대신 tick 수를 센다) |
| 추론 (ONNX Runtime, CPU) | `engine/inference/inference_engine.cpp` | `self._session.run` |
| raw 액션 → 물리량 (`action_mean + scale × raw`) → 모터 명령 | `action.cpp` | `_apply` |
| 관측에 스케일·정규화를 **하지 않는다** (그래프 안에 있으므로) | | 마찬가지 |

관측 함수는 섹터 0 방식(numpy, `MjData` 하나)으로 다시 씁니다. 학습 때의 torch 함수와 **같은 양을 같은 좌표계로** 계산해야 합니다. 이 "같은 계산을 두 언어로 두 번 구현한다" 는 점이 sim-to-real 에서 버그가 가장 자주 나는 곳입니다. World 의 `tests/contract/test_seam3_policy_config.py` 가 World 의 설정과 Core 의 헤더가 어긋나지 않는지 검사하는 이유입니다.

### 문제 17-4. `DeployedController`

- `OBS_TERMS`: 이름 → `fn(m, d, ctrl) -> np.ndarray`. `ctrl` 은 제어기 자신이며 `ctrl.goal_xy`, `ctrl.last_raw_action`, `ctrl.body_id`, `ctrl.qadr`, `ctrl.vadr` 를 쓸 수 있습니다. 네 항을 채웁니다. (각속도는 body → 세계 변환이 필요합니다. 섹터 0 Part 2.)
- `__init__`: 차원 검사. 맞지 않으면 `ValueError`.
- `tick(m, d)`: **물리 스텝마다** 불립니다. `decimation` tick 마다 한 번만 관측을 만들고 추론합니다. 토크는 매 tick 써넣습니다.


In [ ]:
def _np_quat_rotate(q, v):
    """(주어진 코드) body → world, numpy. 섹터 0 문제 2-3."""
    w, u = q[0], q[1:]
    t = np.cross(u, v)
    return v + 2.0 * w * t + 2.0 * np.cross(u, t)


def _obs_ball_lin_vel(m, d, c) -> np.ndarray:
    return d.qvel[c.vadr:c.vadr + 2].copy()


def _obs_ball_ang_vel(m, d, c) -> np.ndarray:
    return _np_quat_rotate(d.qpos[c.qadr + 3:c.qadr + 7], d.qvel[c.vadr + 3:c.vadr + 6])


def _obs_goal_rel_pos(m, d, c) -> np.ndarray:
    return c.goal_xy - d.qpos[c.qadr:c.qadr + 2]


def _obs_last_action(m, d, c) -> np.ndarray:
    return c.last_raw_action.copy()


OBS_TERMS = {"ball_lin_vel": _obs_ball_lin_vel, "ball_ang_vel": _obs_ball_ang_vel,
             "goal_rel_pos": _obs_goal_rel_pos, "last_action": _obs_last_action}


class DeployedController:
    """ONNX 파일 하나로 공을 제어한다. DIDEN_Core 의 RlController 축소판."""

    def __init__(self, onnx_path: str, m: mujoco.MjModel, body_name: str = "ball/sphere"):
        self._session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        meta = self._session.get_modelmeta().custom_metadata_map
        if POLICY_CONFIG_METADATA_KEY not in meta:
            raise ValueError("policy_config 메타데이터가 없는 ONNX 는 실행하지 않는다")
        self.config = yaml.safe_load(meta[POLICY_CONFIG_METADATA_KEY])["rl"]
        obs_dim = sum(t["dim"] for t in self.config["observations"])
        act_dim = sum(t["dim"] for t in self.config["actions"])
        in_dim, out_dim = self._session.get_inputs()[0].shape[1], self._session.get_outputs()[0].shape[1]
        if obs_dim != in_dim:
            raise ValueError(f"관측 차원 불일치: policy_config {obs_dim} vs ONNX 입력 {in_dim}")
        if act_dim != out_dim:
            raise ValueError(f"액션 차원 불일치: policy_config {act_dim} vs ONNX 출력 {out_dim}")
        unknown = [t["name"] for t in self.config["observations"] if t["name"] not in OBS_TERMS]
        if unknown:
            raise ValueError(f"모르는 관측 항: {unknown}")
        if abs(m.opt.timestep - self.config["physics_dt"]) > 1e-9:
            raise ValueError(f"물리 주기 불일치: 모델 {m.opt.timestep} vs policy_config {self.config['physics_dt']}")
        self.body_id = m.body(body_name).id
        joint = m.joint(m.body_jntadr[self.body_id])
        self.qadr, self.vadr = int(joint.qposadr[0]), int(joint.dofadr[0])
        self.goal_xy = np.zeros(2)
        self.last_raw_action = np.zeros(act_dim, dtype=np.float32)
        self.tick_count = 0
        self.inference_count = 0

    def _build_obs(self, m, d) -> np.ndarray:
        parts = [np.asarray(OBS_TERMS[t["name"]](m, d, self), dtype=np.float32) for t in self.config["observations"]]
        return np.concatenate(parts)[None, :]

    def _apply(self, d) -> None:
        ctrl = self.config["control"]
        a = self.last_raw_action
        if ctrl["action_clip"] is not None:
            a = np.clip(a, -ctrl["action_clip"], ctrl["action_clip"])
        a = a * ctrl["action_scale"]
        d.xfrc_applied[self.body_id, 3:6] = [-a[1], a[0], 0.0]

    def tick(self, m, d) -> None:
        """물리 스텝마다 mj_step 직전에 부른다."""
        if self.tick_count % self.config["decimation"] == 0:
            obs = self._build_obs(m, d)
            self.last_raw_action = self._session.run(None, {"input": obs})[0][0]
            self.inference_count += 1
        self._apply(d)
        self.tick_count += 1


In [ ]:
def _deploy_run(friction: float, goal_xy, seconds: float = 4.0, record_frames: bool = False):
    m = build_ball_scene(ball_friction=friction).compile()
    d = mujoco.MjData(m)
    ctrl = DeployedController(ONNX_PATH, m)
    ctrl.goal_xy = np.asarray(goal_xy, dtype=float)
    d.mocap_pos[0] = [*goal_xy, 0.0]
    frames = []
    with mujoco.Renderer(m, 320, 320) as renderer:
        for i in range(round(seconds / m.opt.timestep)):
            ctrl.tick(m, d)
            mujoco.mj_step(m, d)
            if record_frames and i % 8 == 0:
                renderer.update_scene(d, camera=make_camera(lookat=(0, 0, 0), distance=8.5, azimuth=90, elevation=-89))
                frames.append(renderer.render())
    dist = float(np.linalg.norm(ctrl.goal_xy - d.qpos[:2]))
    return dist, float(np.linalg.norm(d.qvel[:2])), ctrl, frames

_dist, _speed, _ctrl, _frames = _deploy_run(friction=0.6, goal_xy=(2.0, -1.5), record_frames=True)
print(f"   4초 뒤: 목적지까지 {_dist:.3f} m, 속력 {_speed:.3f} m/s, 물리 tick {_ctrl.tick_count}, 추론 {_ctrl.inference_count} 회")
check("추론 횟수 = tick / decimation", _ctrl.tick_count == 800 and _ctrl.inference_count == 200)
check("ONNX 만으로 목적지에 도착해 멈춤", _dist < 0.25 and _speed < 0.3)
for _mu in (0.3, 1.0):
    _dist, _speed, _, _ = _deploy_run(friction=_mu, goal_xy=(-2.5, 2.0))
    check(f"마찰 {_mu}: 거리 {_dist:.3f} m, 속력 {_speed:.3f} m/s", _dist < 0.25 and _speed < 0.3)
media.show_video(_frames, fps=25)

# 약속이 어긋난 파일은 거부해야 한다
_proto = onnx.load(ONNX_PATH)
_bad = build_policy_config(env)
_bad["rl"]["observations"].pop(1)                                     # 관측 항 하나를 빼먹은 설정
for _p in _proto.metadata_props:
    if _p.key == POLICY_CONFIG_METADATA_KEY:
        _p.value = yaml.safe_dump(_bad, sort_keys=False)
onnx.save(_proto, str(OUT_DIR / "bad_policy.onnx"))
try:
    DeployedController(str(OUT_DIR / "bad_policy.onnx"), build_ball_scene().compile())
    _raised = False
except ValueError as e:
    _raised = True
    print("   거부 사유:", e)
check("관측 차원이 맞지 않는 ONNX 는 거부", _raised)


### 먼저 예상해 보기 → 직접 해 보기

`OBS_TERMS` 를 고쳐서 일부러 약속을 어겨 봅니다. 각각 무슨 일이 생길지 예상한 뒤 `_deploy_run` 으로 확인합니다.

1. `ball_ang_vel` 에서 body → 세계 변환을 빼고 `d.qvel` 의 값을 그대로 쓰면? (차원은 맞으므로 `DeployedController` 는 거부하지 못합니다)
2. `goal_rel_pos` 의 부호를 반대로(공 − 목적지) 하면?
3. `policy_config` 의 `decimation` 을 1 로 바꾼 파일을 만들어 돌리면? (정책이 200 Hz 로 돕니다. `physics_dt` 검사는 통과합니다)
4. `control.action_scale` 을 2.0 으로 바꾸면?

1번과 3번은 "돌아는 가는데 미묘하게 나쁜" 종류입니다. 차원 검사로는 잡히지 않고, 실제 로봇에서는 "sim 에서는 잘 되던 정책이 조금 불안하다" 로 나타납니다. Core 에 정책 입출력을 기록하는 로그(`tools/log_viewer` 의 `rl_io` 뷰어)와, 그 로그를 World 형식으로 되돌려 시뮬레이션과 비교하는 내보내기가 있는 이유입니다.

**실제 코드에서는**: Core 의 관측 항에는 이력(오래된 것부터 / 최신부터 두 가지 순서), EMA·노치 필터, 명령의 데드존과 한계 자르기가 있고, 액션에는 자석 게이트(문턱값, 발별·전체 쿨다운), NaN 일 때 직전 목표 유지, 정책 전환 시 블렌딩이 있습니다. 그리고 우리 `last_action` 은 raw 액션이지만 Core 의 `previous_joint_actions` 는 **직전 관절 목표 각도** 입니다. 같은 이름처럼 보여도 무엇을 담는지는 코드를 열어 봐야 압니다. 이 부분은 **섹터 4** 에서 C++ 퀴즈로 직접 구현합니다(`study/s4_core_rl_quiz/`).


## 섹터 1 을 마치며

만든 것을 World 의 파일과 나란히 놓으면 이렇습니다.

| 우리가 만든 것 | 줄 수(대략) | World 의 대응 파일 | 줄 수 |
|---|---|---|---|
| `BatchedSim` | 30 | `mjlab.sim.Simulation` + `diden_mjlab/sim/sim.py` | 수백 |
| `SimAdapter` | 70 | `diden_common/sim_adapter.py` + `sim_adapter_ports.py` + `diden_mjlab/sim_adapter.py` | 2000+ |
| 매니저 6 개 | 250 | `diden_env/managers/*.py` | 3500 |
| MDP 함수 15 개 | 80 | `diden_env/mdp/`, `diden_domain/mdp/` | 수천 |
| `ManagerBasedRlEnv` | 90 | `manager_based_env.py` + `manager_based_rl_env.py` + `mjwarp_env.py` | 1600 |
| `ActorCritic`, `RolloutStorage`, `PPO`, `OnPolicyRunner` | 250 | `diden_rl/` | 5000+ |
| `export_policy`, `DeployedController` | 90 | `tools/deploy/`, `diden_env/deploy/`; Core `rl_controller/` | 수천 |

줄 수가 수십 배 차이 나는 이유는 대부분 다음 다섯 가지입니다. 실제 코드를 읽다가 낯선 덩어리를 만나면 이 중 어디에 속하는지부터 가려 봅니다.

1. **로봇이라서**: 관절·PD·접촉·보행 위상·자석. (섹터 2)
2. **sim-to-real 이라서**: 센서 지연, 액션 지연, 편향, 이력, 추정기, 더 많은 랜덤화.
3. **엔진이 셋이라서**: 포트로 나눈 `SimAdapter`, 엔진별 구현, 계약 테스트.
4. **크게 돌려야 해서**: 캐시, CUDA 그래프, 로그 조절, NaN 감시, 원격 학습.
5. **여러 과제를 한 코드로**: regime, 커리큘럼, 설정 덮어쓰기, 작업 등록.

### 스스로 답해 볼 질문 (`study/QUESTION_TREE.md` 의 Q1 가지)

1. `step()` 안에서 보상 → 리셋 → 관측 순서를 바꾸면 각각 무슨 일이 생기는가?
2. actor 와 critic 의 관측을 다르게 주는 것이 왜 허용되는가? critic 에만 준 정보가 정책에 "새어 들어갈" 길이 있는가?
3. 시간 초과와 실패를 구분하지 않으면 가치 함수는 무엇을 배우게 되는가?
4. 보상을 `(Σ add) × (Π mul)` 로 묶는 설계의 단점은 무엇인가? (힌트: add 가 0 근처일 때 mul 항들의 기울기)
5. 관측 정규화 통계를 ONNX 안에 넣지 않고 Core 가 따로 갖고 있게 했다면 어떤 사고가 날 수 있는가?
6. `decimation` 을 바꾸면 다시 맞춰야 하는 값들은 무엇인가? (보상의 `dt`, 에피소드 길이, 이력 간격, 액션 변화율 벌점 …)
7. 환경 수를 4096 에서 64 로 줄이고 스텝 수를 24 에서 1536 으로 늘리면(표본 수는 같음) 학습이 같을까?

### 다음

- **캡스톤 (선택)**: 이 환경을 **진짜 DIDEN_World 위에** 작업(task)으로 등록해 `dw train` 으로 학습시켜 봅니다. 우리가 만든 것과 이름이 같은 진짜 클래스(`ObsTerm`, `RewTerm`, `EventTerm`, `ManagerBasedRLEnvCfg`)로 설정만 쓰면 되고, 막히는 지점이 곧 섹터 2 에서 배울 내용입니다.
- **섹터 2 — Spider 보행 MDP**: 관측·액션·보상·이벤트 항을 실제 값 그대로 직접 구현하고 World 의 함수와 출력이 일치하는지 채점합니다.
- **섹터 4 — Core RL 제어 스택**: Part 17 의 `DeployedController` 를 C++ 로, Core 의 실제 동작 그대로.
